# NB-02_proceso_ejecucion_gc_cgr

Process Flow SAS: **Proceso Ejecución GC CGR** — `PFD-wyxachgCf8TVObWz`

In [ ]:
# ========= Parámetros =========
# Variables macro del SAS original. El .egp NO las define (venían del
# entorno SAS): su valor sale de la entrevista B4 o de
# project_config.yaml → run.macro_params, o se inyecta acá
# (celda 'parameters' de papermill).

Analista = None  # &Analista — nadie declaró su valor
Year = None  # &Year — nadie declaró su valor

faltantes = [n for n, v in {"Analista": Analista, "Year": Year}.items() if v is None]
if faltantes:
    raise ValueError(f"Parámetros sin valor: {faltantes}")

In [ ]:
# ========= Celda 1: Configuración =========
import pandas as pd
import numpy as np
import os
import sqlalchemy
import datetime
from pathlib import Path
from sqlalchemy import text
from datetime import date
import pathlib

# Conexión a BD — la fija el orquestador con SASMIG_DB_URL; si no está,
# se usa la conexión por defecto del proyecto (SUPUESTO: verificar servidor
# y base antes de correr contra datos reales).
engine = sqlalchemy.create_engine(
    os.environ.get("SASMIG_DB_URL", 'mssql+pyodbc:///?odbc_connect=DRIVER={ODBC Driver 17 for SQL Server};SERVER=tcp:PLATDAT,1433;DATABASE=PYDCNIGOB;Authentication=ActiveDirectoryIntegrated;Encrypt=yes;TrustServerCertificate=no')
)

# Logging liviano de resultados — aprobado en la entrevista (Fase 4)
_LOG_PATH = Path("log") / "NB-02_proceso_ejecucion_gc_cgr.log"
_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
def _log(label, value=None):
    """Una línea por celda: imprime y persiste. Jamás rompe la corrida."""
    try:
        if hasattr(value, "shape"):
            detail = f"{value.shape[0]} filas x {value.shape[1]} cols"
        elif isinstance(value, int):
            detail = f"{value} filas"
        elif value is None:
            detail = "ok"
        else:
            detail = str(value)
        line = f"[{datetime.datetime.now():%Y-%m-%d %H:%M:%S}] {label}: {detail}"
        print(line)
        with open(_LOG_PATH, "a", encoding="utf-8") as fh:
            fh.write(line + "\n")
    except Exception:
        pass  # el log nunca puede tumbar el notebook
with open(_LOG_PATH, "a", encoding="utf-8") as _fh:
    _fh.write(f"\n=== corrida {datetime.datetime.now():%Y-%m-%d %H:%M:%S} ===\n")


## 1-Check Compara PACAPR

*confianza: high · verificador: approve · SAS: PROC SQL CREATE TABLE con LEFT JOIN y DISTINCT*

In [ ]:
# ========= 1-Check Compara PACAPR =========
sql_compara_pacapr = """
SELECT DISTINCT t1.C_ENTIDAD,
       t2.C_ENTIDAD AS C_ENTIDAD_ANTIG,
       t1.C_PARTIDA,
       t1.N_PARTIDA,
       t2.N_PARTIDA AS N_PARTIDA_ANTIG,
       t1.C_CAPITULO,
       t1.N_CAPITULO,
       t2.N_CAPITULO AS N_CAPITULO_ANTIG,
       t1.C_PROGRAMA,
       t1.N_PROGRAMA,
       t2.N_PROGRAMA AS N_PROGRAMA_ANTIG
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.ENTIDADES t2
  ON t1.C_PARTIDA = t2.C_PARTIDA
  AND t1.C_CAPITULO = t2.C_CAPITULO
  AND t1.C_PROGRAMA = t2.C_PROGRAMA
WHERE t1.AÑO = :anio
"""
compara_pacapr = pd.read_sql(text(sql_compara_pacapr), engine, params={"anio": Year})
_log("compara_pacapr", compara_pacapr)


## 2-Check Compara C_ENTIDAD

*confianza: medium · verificador: approve · SAS: PROC SQL CREATE TABLE con LEFT JOIN y SELECT DISTINCT*

In [ ]:
# ========= 2-Check Compara C_ENTIDAD =========
sql_compara_entidad = """
SELECT DISTINCT t1.C_ENTIDAD,
       t1.C_PARTIDA,
       t2.C_PARTIDA AS C_PARTIDA_ANTIG,
       t1.N_PARTIDA,
       t2.N_PARTIDA AS N_PARTIDA_ANTIG,
       t1.C_CAPITULO,
       t2.C_CAPITULO AS C_CAPITULO_ANTIG,
       t1.N_CAPITULO,
       t2.N_CAPITULO AS N_CAPITULO_ANTIG,
       t1.C_PROGRAMA,
       t2.C_PROGRAMA AS C_PROGRAMA_ANTIG,
       t1.N_PROGRAMA,
       t2.N_PROGRAMA AS N_PROGRAMA_ANTIG
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.ENTIDADES t2 ON (t1.C_ENTIDAD = t2.C_ENTIDAD)
WHERE t1.ANIO = :year
"""
compara_entidad = pd.read_sql(text(sql_compara_entidad), engine, params={"year": Year})
_log("compara_entidad", compara_entidad)


## 3a-Check Nvas Entidades

*confianza: high · verificador: approve · SAS: PROC SQL CREATE TABLE con LEFT JOIN + IS NULL (anti-join) filtrado por año*

In [ ]:
# ========= 3a-Check Nvas Entidades =========
sql_check_entidades = """
SELECT DISTINCT t1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.N_PARTIDA,
       t1.N_CAPITULO,
       t1.N_PROGRAMA
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.ENTIDADES t2
  ON t1.C_PARTIDA = t2.C_PARTIDA
 AND t1.C_CAPITULO = t2.C_CAPITULO
 AND t1.C_PROGRAMA = t2.C_PROGRAMA
WHERE t2.C_ENTIDAD IS NULL AND t1.AÑO = :year
"""
check_entidades = pd.read_sql(text(sql_check_entidades), engine, params={"year": Year})
_log("check_entidades", check_entidades)


## 3b-Clasif Nvas Entidades

*confianza: medium · verificador: approve · SAS: PROC SQL CREATE TABLE (LEFT JOIN + anti-join) + PROC DATASETS APPEND FORCE*

In [ ]:
# ========= 3b-Clasif Nvas Entidades =========
sql_nuevas_entidades = """
SELECT DISTINCT t1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.N_PARTIDA,
       t1.C_CAPITULO,
       t1.N_CAPITULO,
       t1.C_PROGRAMA,
       t1.N_PROGRAMA
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.ENTIDADES t2
  ON t1.C_PARTIDA = t2.C_PARTIDA
 AND t1.C_CAPITULO = t2.C_CAPITULO
 AND t1.C_PROGRAMA = t2.C_PROGRAMA
WHERE t2.C_ENTIDAD IS NULL AND t1.AÑO = :year
"""
nuevas_entidades = pd.read_sql(text(sql_nuevas_entidades), engine, params={"year": Year})
_log("nuevas_entidades", nuevas_entidades)


In [ ]:
if not nuevas_entidades.empty:
    nuevas_entidades.to_sql("ENTIDADES", engine, schema="dbo", if_exists="append", index=False)
    _log("APPEND GOBGENER.dbo.ENTIDADES", len(nuevas_entidades))
else:
    _log("APPEND GOBGENER.dbo.ENTIDADES", 0)


## 4a-Check Analiticos Predef_mod

*confianza: medium · verificador: approve · SAS: PROC SQL CREATE TABLE con LEFT JOIN, GROUP BY, MAX por clave y UPDATE/DELETE condicionales*

In [ ]:
# ========= 4a-Check Analiticos Predef_mod =========
# I. Extrae ejecución CGR del año del proyecto (filtro en el WHERE, mínimo necesario)
sql_ejec_cgr = """
SELECT *
FROM GOBGENER.dbo.EJECUCION
WHERE AÑO = :anio
"""
ejec_cgr = pd.read_sql(text(sql_ejec_cgr), engine, params={"anio": Year})
_log("ejec_cgr", ejec_cgr)


In [ ]:
# Determina analíticos que no están en la tabla fija y requieren clasificación
sql_cr18 = """
SELECT T_CUENTA, C_CUENTA, C_ANALITICO
FROM GOBGENER.dbo.CR18_T_SCN_1
"""
cr18_t_scn_1 = pd.read_sql(text(sql_cr18), engine)

merged = ejec_cgr.merge(
    cr18_t_scn_1,
    left_on=["C_TIPO_CUENTA", "C_CUENTA", "C_ANALITICO"],
    right_on=["T_CUENTA", "C_CUENTA", "C_ANALITICO"],
    how="left",
    suffixes=("", "_cr18"),
)

# T2.C_ANALITICO AS C_ANALIT: como el join es por C_ANALITICO, el valor coincidente
# (cuando hay match) es el mismo C_ANALITICO; si no hay match, es NaN/""
merged["C_ANALIT"] = np.where(merged["T_CUENTA"].notna(), merged["C_ANALITICO"], pd.NA)

filtro = (merged["T_CUENTA"].isna() | (merged["C_ANALIT"] == "")) & (~merged["C_CUENTA"].isin(["05", "13", "24", "33"]))
base_faltante = merged[filtro].copy()

base_faltante["KEY_CORTA"] = (
    base_faltante["C_CUENTA"].astype(str) + "_" + base_faltante["C_ITEM"].astype(str) + "_" + base_faltante["C_ASIGNACION"].astype(str)
)
base_faltante["KEY_CORTA_2"] = (
    base_faltante["C_CUENTA"].astype(str) + "_" + base_faltante["C_ITEM"].astype(str) + "_" + base_faltante["C_ASIGNACION"].astype(str).str[:2]
)

analit_fijo_faltante = (
    base_faltante.groupby(
        ["C_TIPO_CUENTA", "C_CUENTA", "C_ITEM", "C_ASIGNACION", "C_ANALITICO", "N_ANALITICO", "C_ANALIT", "KEY_CORTA", "KEY_CORTA_2"],
        dropna=False,
    )["DEVENG"]
    .count()
    .reset_index()
    .rename(columns={"DEVENG": "ID"})
)
_log("analit_fijo_faltante", analit_fijo_faltante)


In [ ]:
# II. Propone clasificaciones usando frecuencia máxima por KEY_CORTA (CUENTA-ITEM-ASIGNACION)
sql_cr18_full = """
SELECT T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_SCN, C_TRANSACCION, T_TRANSACCION,
       N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, FECHA
FROM GOBGENER.dbo.CR18_T_SCN_1
"""
cr18_full = pd.read_sql(text(sql_cr18_full), engine)

cr18_full["KEY_CORTA"] = (
    cr18_full["C_CUENTA"].astype(str) + "_" + cr18_full["C_ITEM"].astype(str) + "_" + cr18_full["C_ASIGNACION"].astype(str)
)

cruce_fijo = (
    cr18_full.groupby(
        ["T_CUENTA", "C_CUENTA", "C_ITEM", "C_ASIGNACION", "C_SCN", "C_TRANSACCION", "N_CUENTA", "C_CAGENTE", "C_CUP", "C_ACTIVO", "T_TRANSACCION", "KEY_CORTA"],
        dropna=False,
    )["FECHA"]
    .count()
    .reset_index()
    .rename(columns={"FECHA": "ID"})
)
_log("cruce_fijo", cruce_fijo)


In [ ]:
cruce_fijo_max = cruce_fijo.groupby("KEY_CORTA", dropna=False)["ID"].max().reset_index().rename(columns={"ID": "ID_MAX"})

cruce_prop = cruce_fijo.merge(cruce_fijo_max, on="KEY_CORTA", how="inner")
cruce_prop = cruce_prop[cruce_prop["ID"] == cruce_prop["ID_MAX"]].drop(columns=["ID_MAX"])
cruce_prop = cruce_prop.drop_duplicates(subset=["KEY_CORTA"], keep="first")
_log("cruce_prop", cruce_prop)


In [ ]:
# Genera claves con KEY_CORTA_2 (asignación truncada a 2 dígitos)
cr18_full["KEY_CORTA_2"] = (
    cr18_full["C_CUENTA"].astype(str) + "_" + cr18_full["C_ITEM"].astype(str) + "_" + cr18_full["C_ASIGNACION"].astype(str).str[:2]
)

cruce_fijo_2 = (
    cr18_full.groupby(
        ["C_SCN", "C_TRANSACCION", "N_CUENTA", "C_CAGENTE", "C_CUP", "C_ACTIVO", "T_TRANSACCION", "KEY_CORTA_2"],
        dropna=False,
    )["FECHA"]
    .count()
    .reset_index()
    .rename(columns={"FECHA": "ID"})
)

cruce_fijo_max_2 = cruce_fijo_2.groupby("KEY_CORTA_2", dropna=False)["ID"].max().reset_index().rename(columns={"ID": "ID_MAX"})

cruce_prop_2 = cruce_fijo_2.merge(cruce_fijo_max_2, on="KEY_CORTA_2", how="inner")
cruce_prop_2 = cruce_prop_2[cruce_prop_2["ID"] == cruce_prop_2["ID_MAX"]].drop(columns=["ID_MAX"])
cruce_prop_2 = cruce_prop_2.drop_duplicates(subset=["KEY_CORTA_2"], keep="first")
_log("cruce_prop_2", cruce_prop_2)


In [ ]:
# Asigna clasificaciones propuestas usando KEY_CORTA (clave 1)
ANALISTA = Analista.upper()  # upcase("&Analista")
hoy = date.today()

analit_fijo_clas = analit_fijo_faltante.merge(
    cruce_prop[["KEY_CORTA", "C_SCN", "C_TRANSACCION", "T_TRANSACCION", "N_CUENTA", "C_CAGENTE", "C_CUP", "C_ACTIVO"]],
    on="KEY_CORTA",
    how="left",
)

analit_fijo_clas = analit_fijo_clas.rename(columns={"C_TIPO_CUENTA": "T_CUENTA", "N_ANALITICO": "ANALITICO"})
analit_fijo_clas["FECHA"] = hoy
analit_fijo_clas["ANALISTA"] = ANALISTA
analit_fijo_clas = analit_fijo_clas[[
    "T_CUENTA", "C_CUENTA", "C_ITEM", "C_ASIGNACION", "C_ANALITICO", "ANALITICO",
    "C_SCN", "C_TRANSACCION", "T_TRANSACCION", "N_CUENTA", "C_CAGENTE", "C_CUP", "C_ACTIVO",
    "FECHA", "ANALISTA", "KEY_CORTA", "KEY_CORTA_2",
]]
_log("analit_fijo_clas", analit_fijo_clas)


In [ ]:
# Asigna clasificaciones propuestas usando KEY_CORTA_2, solo donde C_SCN quedó vacío
# (evita duplicar columnas del merge: se dropean las de la izquierda antes de unir)
faltan_scn = analit_fijo_clas[(analit_fijo_clas["C_SCN"].isna()) | (analit_fijo_clas["C_SCN"] == "")].copy()
cols_cruce = ["C_SCN", "C_TRANSACCION", "T_TRANSACCION", "N_CUENTA", "C_CAGENTE", "C_CUP", "C_ACTIVO"]
faltan_scn_base = faltan_scn.drop(columns=cols_cruce)

analit_fijo_clas_2 = faltan_scn_base.merge(
    cruce_prop_2[["KEY_CORTA_2"] + cols_cruce],
    on="KEY_CORTA_2",
    how="left",
)
analit_fijo_clas_2["FECHA"] = hoy
analit_fijo_clas_2["ANALISTA"] = ANALISTA
analit_fijo_clas_2 = analit_fijo_clas_2[[
    "T_CUENTA", "C_CUENTA", "C_ITEM", "C_ASIGNACION", "C_ANALITICO", "ANALITICO",
    "C_SCN", "C_TRANSACCION", "T_TRANSACCION", "N_CUENTA", "C_CAGENTE", "C_CUP", "C_ACTIVO",
    "FECHA", "ANALISTA",
]]
_log("analit_fijo_clas_2", analit_fijo_clas_2)


In [ ]:
# Deja las clasificaciones faltantes como está, para completar a mano
analit_fijo_clas = pd.concat([analit_fijo_clas, analit_fijo_clas_2], ignore_index=True)
analit_fijo_clas = analit_fijo_clas.drop(columns=["KEY_CORTA", "KEY_CORTA_2"], errors="ignore")
analit_fijo_clas = analit_fijo_clas.drop_duplicates(subset=["C_CUENTA", "C_ANALITICO", "C_ITEM"], keep="first")


In [ ]:
# En general las cuentas 07-02 se asocian al P11: autocompleta
mask_p11 = (
    ((analit_fijo_clas["C_SCN"].isna()) | (analit_fijo_clas["C_SCN"] == ""))
    & (analit_fijo_clas["C_CUENTA"] == "07")
    & (analit_fijo_clas["C_ITEM"] == "02")
)
analit_fijo_clas.loc[mask_p11, "C_SCN"] = "P11"
analit_fijo_clas.loc[mask_p11, "C_TRANSACCION"] = "111"
analit_fijo_clas.loc[mask_p11, "T_TRANSACCION"] = "Prod merc"
analit_fijo_clas.loc[mask_p11, "N_CUENTA"] = "Ventas"
analit_fijo_clas.loc[mask_p11, "C_CAGENTE"] = "S1"

# Borra los registros cuando no se pueden clasificar
analit_fijo_clas = analit_fijo_clas[analit_fijo_clas["ANALITICO"] != "SIN DENOMINACION"].reset_index(drop=True)
_log("analit_fijo_clas", analit_fijo_clas)


## 4b-Clasif Analiticos Predef_mod

*confianza: medium · verificador: approve · SAS: PROC SQL DELETE/UPDATE + PROC DATASETS APPEND + PROC SQL CREATE TABLE de verificación*

In [ ]:
# ========= 4b-Clasif Analiticos Predef_mod =========
# I. Anexa clasificaciones faltantes desde paso anterior: DELETE de coincidencias en GOBGENER.CR18_T_SCN_1 usando clave de analit_fijo_clas (nodo anterior en memoria)
keys = analit_fijo_clas[["T_CUENTA", "C_CUENTA", "C_ITEM", "C_ASIGNACION", "C_ANALITICO"]].drop_duplicates()
with engine.begin() as conn:
    keys.to_sql("#tmp_analit_fijo_clas", conn, if_exists="replace", index=False)
    res = conn.execute(text("""
        DELETE FROM GOBGENER.dbo.CR18_T_SCN_1
        WHERE EXISTS (
            SELECT 1 FROM #tmp_analit_fijo_clas t
            WHERE t.T_CUENTA = GOBGENER.dbo.CR18_T_SCN_1.T_CUENTA
              AND t.C_CUENTA = GOBGENER.dbo.CR18_T_SCN_1.C_CUENTA
              AND t.C_ITEM = GOBGENER.dbo.CR18_T_SCN_1.C_ITEM
              AND t.C_ASIGNACION = GOBGENER.dbo.CR18_T_SCN_1.C_ASIGNACION
              AND t.C_ANALITICO = GOBGENER.dbo.CR18_T_SCN_1.C_ANALITICO
        )
    """))
    _log("DELETE GOBGENER.dbo.CR18_T_SCN_1", res.rowcount)


In [ ]:
# UPDATE en memoria: donde C_SCN es vacio se marca 'REV' (equivalente al UPDATE ANALIT_FIJO_CLAS SET C_SCN='REV' WHERE C_SCN='')
analit_fijo_clas.loc[analit_fijo_clas["C_SCN"] == "", "C_SCN"] = "REV"
_log("analit_fijo_clas", analit_fijo_clas)


In [ ]:
# APPEND (PROC DATASETS APPEND FORCE) de analit_fijo_clas hacia GOBGENER.CR18_T_SCN_1 -- acumula, re-ejecutar duplica igual que el SAS original
analit_fijo_clas.to_sql("CR18_T_SCN_1", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND GOBGENER.dbo.CR18_T_SCN_1", len(analit_fijo_clas))


In [ ]:
# Verifica duplicados en la base de clasificaciones fijas: agrupa y cuenta, deja solo los grupos con count != 1
sql_check = """
SELECT T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, COUNT(C_SCN) AS ID
FROM GOBGENER.dbo.CR18_T_SCN_1
GROUP BY T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO
"""
check_analit_fijo = pd.read_sql(text(sql_check), engine)
check_analit_fijo = check_analit_fijo[check_analit_fijo["ID"] != 1]
_log("check_analit_fijo", check_analit_fijo)


In [ ]:
# Verifica que no existan analiticos sin clasificar (C_SCN='REV')
sql_rev = """
SELECT T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, C_SCN
FROM GOBGENER.dbo.CR18_T_SCN_1
WHERE C_SCN = :scn
"""
check_analit_rev = pd.read_sql(text(sql_rev), engine, params={"scn": "REV"})
_log("check_analit_rev", check_analit_rev)


## 5a-Check Analiticos Var_mod

*confianza: medium · verificador: approve · SAS: PROC SQL multi-paso (LEFT JOIN + GROUP BY MAX + UPDATE) y DATA step SET/CATX/DROP para clasificación de analíticos variables*

In [ ]:
# ========= 5a-Check Analiticos Var_mod =========
# Extrae EJEC_CGR del periodo Year desde GOBGENER.EJECUCION (filtro en el WHERE, minimiza extract)
sql_ejec_cgr = """
SELECT *
FROM GOBGENER.dbo.EJECUCION
WHERE AÑO = :anio
"""
ejec_cgr = pd.read_sql(text(sql_ejec_cgr), engine, params={"anio": Year})
_log("ejec_cgr", ejec_cgr)


In [ ]:
# ENT_N: distinct partida/capitulo/programa/entidad desde EJEC_CGR
ent_n = ejec_cgr[["C_PARTIDA", "C_CAPITULO", "C_PROGRAMA", "C_ENTIDAD"]].drop_duplicates()
_log("ent_n", ent_n)


In [ ]:
# ANALIT_SIN_ENTIDAD: analiticos de CR18_T_SCN_2 con ENTIDAD faltante, incorpora C_ENTIDAD via left join con ENT_N
sql_scn2_sin_entidad = """
SELECT *
FROM GOBGENER.dbo.CR18_T_SCN_2
WHERE ENTIDAD IS NULL
"""
scn2_sin_entidad = pd.read_sql(text(sql_scn2_sin_entidad), engine)

analit_sin_entidad = scn2_sin_entidad.merge(
    ent_n,
    on=["C_PARTIDA", "C_CAPITULO", "C_PROGRAMA"],
    how="left",
    suffixes=("", "_ent_n"),
)
# UPDATE ANALIT_SIN_ENTIDAD SET ENTIDAD=C_ENTIDAD (el C_ENTIDAD viene de ENT_N, sufijo _ent_n si colisiona)
col_c_entidad = "C_ENTIDAD_ent_n" if "C_ENTIDAD_ent_n" in analit_sin_entidad.columns else "C_ENTIDAD"
analit_sin_entidad["ENTIDAD"] = analit_sin_entidad[col_c_entidad]
analit_sin_entidad = analit_sin_entidad.drop(columns=[col_c_entidad])
_log("analit_sin_entidad", analit_sin_entidad)


In [ ]:
# Reemplaza en GOBGENER.CR18_T_SCN_2: borra filas sin ENTIDAD y agrega las mismas ya con ENTIDAD asignada (append, replica PROC APPEND)
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM GOBGENER.dbo.CR18_T_SCN_2 WHERE ENTIDAD IS NULL"))
    _log("DELETE GOBGENER.dbo.CR18_T_SCN_2 sin entidad", res.rowcount)
analit_sin_entidad.to_sql("CR18_T_SCN_2", engine, schema="dbo", if_exists="append", index=False)


In [ ]:
# I. Analiticos variables (cuentas 05,13,24,33) faltantes por clasificar del periodo
# LEFT JOIN exige igualdad de C_ANALITICO (match exacto); condicion SAS: T2.C_ANALITICO="" (vacio) tras el join
sql_scn2_full = """
SELECT ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, T_CUENTA, C_CUENTA, C_ANALITICO
FROM GOBGENER.dbo.CR18_T_SCN_2
"""
scn2_full = pd.read_sql(text(sql_scn2_full), engine)

cruce_check = ejec_cgr.merge(
    scn2_full,
    left_on=["C_ENTIDAD", "C_PARTIDA", "C_CAPITULO", "C_PROGRAMA", "C_TIPO_CUENTA", "C_CUENTA", "C_ANALITICO"],
    right_on=["ENTIDAD", "C_PARTIDA", "C_CAPITULO", "C_PROGRAMA", "T_CUENTA", "C_CUENTA", "C_ANALITICO"],
    how="left",
    suffixes=("", "_scn2"),
)
# T2.C_ANALITICO="" en SAS equivale a: no hubo match (NaN) O el C_ANALITICO calzado es cadena vacia
col_c_analitico_scn2 = "C_ANALITICO_scn2" if "C_ANALITICO_scn2" in cruce_check.columns else "C_ANALITICO"
sin_match = cruce_check["ENTIDAD"].isna() | (cruce_check[col_c_analitico_scn2].fillna("") == "")
cruce_check["_sin_match"] = sin_match

filtro = cruce_check["_sin_match"] & cruce_check["C_CUENTA"].isin(["05", "13", "24", "33"])
faltantes = cruce_check[filtro]

analit_entidad_var = (
    faltantes.groupby(
        ["C_ENTIDAD", "C_PARTIDA", "C_CAPITULO", "C_PROGRAMA", "C_TIPO_CUENTA", "C_CUENTA", "C_ITEM", "C_ASIGNACION", "C_ANALITICO", "N_ANALITICO"],
        dropna=False,
    )["DEVENG"].count().reset_index(name="ID")
)
_log("analit_entidad_var", analit_entidad_var)


In [ ]:
# II. Propone clasificaciones para GORE (partida = 88), KEY_1 y KEY_2
def catx(sep, *cols):
    parts = [c.astype(str) for c in cols]
    out = parts[0]
    for p in parts[1:]:
        out = out.str.cat(p, sep=sep, na_rep="")
    return out

analit_entidad_var["KEY_1"] = catx("_", analit_entidad_var["C_PARTIDA"], analit_entidad_var["C_CUENTA"], analit_entidad_var["C_ITEM"], analit_entidad_var["C_ASIGNACION"])
analit_entidad_var["KEY_2"] = catx("_", analit_entidad_var["C_PARTIDA"], analit_entidad_var["C_CUENTA"], analit_entidad_var["C_ASIGNACION"])


In [ ]:
# Trae ya clasificado en CR18_T_SCN_2 para partida 88 (para construir KEY_1/KEY_2 de reutilizacion)
sql_scn2_88 = """
SELECT T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_SCN, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, FECHA, C_PARTIDA
FROM GOBGENER.dbo.CR18_T_SCN_2
WHERE C_PARTIDA = :partida
"""
scn2_88 = pd.read_sql(text(sql_scn2_88), engine, params={"partida": "88"})
scn2_88["KEY_1"] = catx("_", scn2_88["C_PARTIDA"], scn2_88["C_CUENTA"], scn2_88["C_ITEM"], scn2_88["C_ASIGNACION"])

grp_cols = ["T_CUENTA", "C_CUENTA", "C_ITEM", "C_ASIGNACION", "C_SCN", "C_TRANSACCION", "N_CUENTA", "C_CAGENTE", "C_CUP", "C_ACTIVO", "T_TRANSACCION", "KEY_1"]
cruce_var = scn2_88.groupby(grp_cols, dropna=False)["FECHA"].count().reset_index(name="ID")

cruce_var_max = cruce_var.groupby("KEY_1", dropna=False)["ID"].agg(ID_MAX="max", ID="count").reset_index()

cruce_prop = cruce_var.merge(cruce_var_max, on="KEY_1", suffixes=("_x", "_y"))
cruce_prop = cruce_prop[cruce_prop["ID_x"] == cruce_prop["ID_MAX"]]
cruce_prop = cruce_prop.rename(columns={"ID_x": "ID", "ID_y": "ID_COUNT"})
cruce_prop = cruce_prop.drop_duplicates(subset=["KEY_1"], keep="first")
_log("cruce_prop", cruce_prop)


In [ ]:
# KEY_2 equivalente (usa mismos datos scn2_88 ya extraidos)
scn2_88["KEY_2"] = catx("_", scn2_88["C_PARTIDA"], scn2_88["C_CUENTA"], scn2_88["C_ASIGNACION"])
grp_cols_2 = ["T_CUENTA", "C_CUENTA", "C_ITEM", "C_ASIGNACION", "C_SCN", "C_TRANSACCION", "N_CUENTA", "C_CAGENTE", "C_CUP", "C_ACTIVO", "T_TRANSACCION", "KEY_2"]
cruce_var_2 = scn2_88.groupby(grp_cols_2, dropna=False)["FECHA"].count().reset_index(name="ID")
cruce_var_max_2 = cruce_var_2.groupby("KEY_2", dropna=False)["ID"].agg(ID_MAX="max", ID="count").reset_index()
cruce_prop_2 = cruce_var_2.merge(cruce_var_max_2, on="KEY_2", suffixes=("_x", "_y"))
cruce_prop_2 = cruce_prop_2[cruce_prop_2["ID_x"] == cruce_prop_2["ID_MAX"]]
cruce_prop_2 = cruce_prop_2.rename(columns={"ID_x": "ID", "ID_y": "ID_COUNT"})
cruce_prop_2 = cruce_prop_2.drop_duplicates(subset=["KEY_2"], keep="first")
_log("cruce_prop_2", cruce_prop_2)


In [ ]:
# ANALIT_VAR_CLAS: asigna clasificacion propuesta a GORE (partida 88) usando KEY_1
hoy = pd.Timestamp(datetime.date.today())
analista_up = Analista.upper()  # parametro

gore = analit_entidad_var[analit_entidad_var["C_PARTIDA"] == "88"]
analit_var_clas = gore.merge(cruce_prop[["KEY_1", "C_SCN", "C_TRANSACCION", "T_TRANSACCION", "N_CUENTA", "C_CAGENTE", "C_CUP", "C_ACTIVO", "ID_COUNT"]], on="KEY_1", how="left")
analit_var_clas = analit_var_clas.rename(columns={"C_TIPO_CUENTA": "T_CUENTA", "N_ANALITICO": "ANALITICO", "C_ENTIDAD": "ENTIDAD"})
analit_var_clas["FECHA"] = hoy
analit_var_clas["ANALISTA"] = analista_up
cols_1 = ["C_PARTIDA", "C_CAPITULO", "C_PROGRAMA", "T_CUENTA", "C_CUENTA", "C_ITEM", "C_ASIGNACION", "C_ANALITICO", "ANALITICO", "C_SCN", "C_TRANSACCION", "T_TRANSACCION", "N_CUENTA", "C_CAGENTE", "C_CUP", "C_ACTIVO", "ENTIDAD", "FECHA", "ANALISTA", "KEY_1", "KEY_2", "ID_COUNT"]
analit_var_clas = analit_var_clas[cols_1]


In [ ]:
# ANALIT_VAR_CLAS_2: reintenta con KEY_2 los que quedaron sin C_SCN
sin_scn = analit_var_clas[analit_var_clas["C_SCN"].isna() | (analit_var_clas["C_SCN"] == "")]
analit_var_clas_2 = sin_scn.merge(cruce_prop_2[["KEY_2", "C_SCN", "C_TRANSACCION", "T_TRANSACCION", "N_CUENTA", "C_CAGENTE", "C_CUP", "C_ACTIVO", "ID_COUNT"]], on="KEY_2", how="left", suffixes=("", "_k2"))
for c in ["C_SCN", "C_TRANSACCION", "T_TRANSACCION", "N_CUENTA", "C_CAGENTE", "C_CUP", "C_ACTIVO", "ID_COUNT"]:
    col_k2 = f"{c}_k2"
    if col_k2 in analit_var_clas_2.columns:
        analit_var_clas_2[c] = analit_var_clas_2[col_k2]
        analit_var_clas_2 = analit_var_clas_2.drop(columns=[col_k2])
analit_var_clas_2 = analit_var_clas_2.drop(columns=["KEY_1"], errors="ignore")

# elimina de analit_var_clas los que quedaron sin C_SCN (se reemplazan por su version reclasificada)
analit_var_clas = analit_var_clas[~(analit_var_clas["C_SCN"].isna() | (analit_var_clas["C_SCN"] == ""))]

analit_var_clas_gore = pd.concat([analit_var_clas, analit_var_clas_2], ignore_index=True)
analit_var_clas_gore = analit_var_clas_gore.drop(columns=["KEY_1", "KEY_2"], errors="ignore")
_log("analit_var_clas_gore", analit_var_clas_gore)


In [ ]:
# III. Propone clasificaciones para resto de entidades (distinto de GORE): KEY_3 y KEY_4
analit_entidad_var["KEY_3"] = catx("_", analit_entidad_var["C_PARTIDA"], analit_entidad_var["C_CUENTA"], analit_entidad_var["C_ITEM"], analit_entidad_var["C_ASIGNACION"])
analit_entidad_var["KEY_4"] = catx("_", analit_entidad_var["C_PARTIDA"], analit_entidad_var["C_PROGRAMA"], analit_entidad_var["C_CUENTA"], analit_entidad_var["C_ASIGNACION"])
analit_entidad_var = analit_entidad_var.drop(columns=["KEY_1", "KEY_2"], errors="ignore")


In [ ]:
# Trae ya clasificado en CR18_T_SCN_2 para partida != 88 (para KEY_3/KEY_4/KEY_5, se reutiliza en bloque IV)
sql_scn2_no88 = """
SELECT T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_SCN, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, FECHA, C_PARTIDA, ENTIDAD, C_PROGRAMA
FROM GOBGENER.dbo.CR18_T_SCN_2
WHERE C_PARTIDA <> :partida
"""
scn2_no88 = pd.read_sql(text(sql_scn2_no88), engine, params={"partida": "88"})

scn2_no88["KEY_3"] = catx("_", scn2_no88["C_PARTIDA"], scn2_no88["C_CUENTA"], scn2_no88["C_ITEM"], scn2_no88["C_ASIGNACION"])
grp3 = ["T_CUENTA", "C_CUENTA", "C_ITEM", "C_ASIGNACION", "C_SCN", "C_TRANSACCION", "N_CUENTA", "C_CAGENTE", "C_CUP", "C_ACTIVO", "T_TRANSACCION", "KEY_3"]
cruce_var_3 = scn2_no88.groupby(grp3, dropna=False)["FECHA"].count().reset_index(name="ID")
cruce_var_max_3 = cruce_var_3.groupby("KEY_3", dropna=False)["ID"].agg(ID_MAX="max", ID="count").reset_index()
cruce_prop_3 = cruce_var_3.merge(cruce_var_max_3, on="KEY_3", suffixes=("_x", "_y"))
cruce_prop_3 = cruce_prop_3[cruce_prop_3["ID_x"] == cruce_prop_3["ID_MAX"]]
cruce_prop_3 = cruce_prop_3.rename(columns={"ID_x": "ID", "ID_y": "ID_COUNT"}).drop_duplicates(subset=["KEY_3"], keep="first")


In [ ]:
scn2_no88["KEY_4"] = catx("_", scn2_no88["C_PARTIDA"], scn2_no88["C_PROGRAMA"], scn2_no88["C_CUENTA"], scn2_no88["C_ASIGNACION"])
grp4 = ["T_CUENTA", "C_CUENTA", "C_ITEM", "C_ASIGNACION", "C_SCN", "C_TRANSACCION", "N_CUENTA", "C_CAGENTE", "C_CUP", "C_ACTIVO", "T_TRANSACCION", "KEY_4"]
cruce_var_4 = scn2_no88.groupby(grp4, dropna=False)["FECHA"].count().reset_index(name="ID")
cruce_var_max_4 = cruce_var_4.groupby("KEY_4", dropna=False)["ID"].agg(ID_MAX="max", ID="count").reset_index()
cruce_prop_4 = cruce_var_4.merge(cruce_var_max_4, on="KEY_4", suffixes=("_x", "_y"))
cruce_prop_4 = cruce_prop_4[cruce_prop_4["ID_x"] == cruce_prop_4["ID_MAX"]]
cruce_prop_4 = cruce_prop_4.rename(columns={"ID_x": "ID", "ID_y": "ID_COUNT"}).drop_duplicates(subset=["KEY_4"], keep="first")


In [ ]:
# ANALIT_VAR_CLAS_3 (KEY_3) y ANALIT_VAR_CLAS_4 (KEY_4) sobre resto (partida != 88)
resto = analit_entidad_var[analit_entidad_var["C_PARTIDA"] != "88"]
analit_var_clas_3 = resto.merge(cruce_prop_3[["KEY_3", "C_SCN", "C_TRANSACCION", "T_TRANSACCION", "N_CUENTA", "C_CAGENTE", "C_CUP", "C_ACTIVO", "ID_COUNT"]], on="KEY_3", how="left")
analit_var_clas_3 = analit_var_clas_3.rename(columns={"C_TIPO_CUENTA": "T_CUENTA", "N_ANALITICO": "ANALITICO", "C_ENTIDAD": "ENTIDAD"})
analit_var_clas_3["FECHA"] = hoy
analit_var_clas_3["ANALISTA"] = analista_up
cols_3 = ["C_PARTIDA", "C_CAPITULO", "C_PROGRAMA", "T_CUENTA", "C_CUENTA", "C_ITEM", "C_ASIGNACION", "C_ANALITICO", "ANALITICO", "C_SCN", "C_TRANSACCION", "T_TRANSACCION", "N_CUENTA", "C_CAGENTE", "C_CUP", "C_ACTIVO", "ENTIDAD", "FECHA", "ANALISTA", "KEY_3", "KEY_4", "ID_COUNT"]
analit_var_clas_3 = analit_var_clas_3[cols_3]

analit_var_clas_4 = analit_var_clas_3.merge(cruce_prop_4[["KEY_4", "C_SCN", "C_TRANSACCION", "T_TRANSACCION", "N_CUENTA", "C_CAGENTE", "C_CUP", "C_ACTIVO", "ID_COUNT"]], on="KEY_4", how="left", suffixes=("", "_k4"))
for c in ["C_SCN", "C_TRANSACCION", "T_TRANSACCION", "N_CUENTA", "C_CAGENTE", "C_CUP", "C_ACTIVO", "ID_COUNT"]:
    col_k4 = f"{c}_k4"
    if col_k4 in analit_var_clas_4.columns:
        analit_var_clas_4[c] = analit_var_clas_4[col_k4]
        analit_var_clas_4 = analit_var_clas_4.drop(columns=[col_k4])
analit_var_clas_4 = analit_var_clas_4[analit_var_clas_4["C_SCN"].isna() | (analit_var_clas_4["C_SCN"] == "")]

analit_var_clas_3 = analit_var_clas_3[~(analit_var_clas_3["C_SCN"].isna() | (analit_var_clas_3["C_SCN"] == ""))]

analit_var_clas_resto = pd.concat([analit_var_clas_3, analit_var_clas_4], ignore_index=True)
analit_var_clas_resto = analit_var_clas_resto.drop(columns=["KEY_3", "KEY_4"], errors="ignore")
_log("analit_var_clas_resto", analit_var_clas_resto)


In [ ]:
# IV. Propone clasificacion para resto de entidades faltantes del paso anterior, cuando se mantiene ENTIDAD pero cambia PARTIDA (KEY_5)
analit_var_clas_resto_f = analit_var_clas_resto[analit_var_clas_resto["C_SCN"] == ""].copy()
analit_var_clas_resto_f["KEY_5"] = catx("_", analit_var_clas_resto_f["ENTIDAD"], analit_var_clas_resto_f["C_CUENTA"], analit_var_clas_resto_f["C_ITEM"], analit_var_clas_resto_f["C_ASIGNACION"])

scn2_no88["KEY_5"] = catx("_", scn2_no88["ENTIDAD"], scn2_no88["C_CUENTA"], scn2_no88["C_ITEM"], scn2_no88["C_ASIGNACION"])
grp5 = ["T_CUENTA", "C_CUENTA", "C_ITEM", "C_ASIGNACION", "C_SCN", "C_TRANSACCION", "N_CUENTA", "C_CAGENTE", "C_CUP", "C_ACTIVO", "T_TRANSACCION", "KEY_5"]
cruce_var_5 = scn2_no88.groupby(grp5, dropna=False)["FECHA"].count().reset_index(name="ID")
cruce_var_max_5 = cruce_var_5.groupby("KEY_5", dropna=False)["ID"].agg(ID_MAX="max", ID="count").reset_index()
cruce_prop_5 = cruce_var_5.merge(cruce_var_max_5, on="KEY_5", suffixes=("_x", "_y"))
cruce_prop_5 = cruce_prop_5[cruce_prop_5["ID_x"] == cruce_prop_5["ID_MAX"]]
cruce_prop_5 = cruce_prop_5.rename(columns={"ID_x": "ID", "ID_y": "ID_COUNT"}).drop_duplicates(subset=["KEY_5"], keep="first")

analit_var_clas_5 = analit_var_clas_resto_f.merge(
    cruce_prop_5[["KEY_5", "C_SCN", "C_TRANSACCION", "T_TRANSACCION", "N_CUENTA", "C_CAGENTE", "C_CUP", "C_ACTIVO", "ID_COUNT"]],
    on="KEY_5", how="left", suffixes=("", "_k5"),
)
for c in ["C_SCN", "C_TRANSACCION", "T_TRANSACCION", "N_CUENTA", "C_CAGENTE", "C_CUP", "C_ACTIVO", "ID_COUNT"]:
    col_k5 = f"{c}_k5"
    if col_k5 in analit_var_clas_5.columns:
        analit_var_clas_5[c] = analit_var_clas_5[col_k5]
        analit_var_clas_5 = analit_var_clas_5.drop(columns=[col_k5])
analit_var_clas_5 = analit_var_clas_5.drop(columns=["KEY_5"], errors="ignore")
_log("analit_var_clas_5", analit_var_clas_5)


In [ ]:
# Une tablas finales de clasificaciones propuestas: GORE + RESTO (solo C_SCN != '') + KEY_5
analit_var_clas_resto_final = analit_var_clas_resto[analit_var_clas_resto["C_SCN"] != ""]

check_transvar = pd.concat([analit_var_clas_gore, analit_var_clas_resto_final, analit_var_clas_5], ignore_index=True)
check_transvar["OBS"] = "Ingresar texto si aplica"
check_transvar = check_transvar.drop(columns=["ID_COUNT"], errors="ignore")

# Ajustes puntuales replicando los UPDATE de SAS (valores hardcodeados del proceso original)
mask_d63 = check_transvar["C_SCN"].isin(["D63-P2", "D63-P"])
check_transvar.loc[mask_d63, "OBS"] = "CR18_difcoy"
check_transvar.loc[mask_d63, "C_ACTIVO"] = "D63"

check_transvar.loc[check_transvar["C_SCN"] == "P3", "OBS"] = "CR18_difact"

mask_p2 = (check_transvar["C_SCN"] == "P2") & (check_transvar["C_CUP"].isin(["EEPP", "SSPP"]))
check_transvar.loc[mask_p2, "OBS"] = "CR18_difact"
_log("check_transvar", check_transvar)


In [ ]:
# Exporta propuesta de clasificaciones a Excel para revision. Ruta original UNC via PC Files Server (EXCELCS/SERVER_NAME);
# se traduce a ruta relativa del workspace (M-001) sin servidor intermedio ni credenciales replicadas
ruta_salida = Path("salidas") / "CHECK_TRANSVAR_PROP.xlsb"
ruta_salida.parent.mkdir(parents=True, exist_ok=True)
check_transvar.to_excel(ruta_salida, sheet_name="PROPUESTA", index=False)
_log("export CHECK_TRANSVAR_PROP", len(check_transvar))


## 5b-Clasif Analiticos Var_mod

*confianza: medium · verificador: approve · SAS: PROC IMPORT EXCELCS + PROC SQL DELETE/UPDATE/ALTER + DELETE...WHERE EXISTS + PROC DATASETS APPEND + PROC SQL CREATE TABLE con GROUP BY*

In [ ]:
# ========= 5b-Clasif Analiticos Var_mod =========
# Import del archivo Excel binario (.xlsb) vía bridge PC Files Server -> se lee directo por ruta relativa (M-001)
ruta_check_transvar_p = Path("insumos") / "GobiernoGeneral" / "CHECK_TRANSVAR_PROP.xlsb"
check_transvar_p = pd.read_excel(ruta_check_transvar_p, sheet_name="PROPUESTA", engine="pyxlsb")


In [ ]:
# Elimina filas totalmente vacías en las claves de partida/capítulo/programa/analítico
check_transvar_p = check_transvar_p[
    ~(
        (check_transvar_p["C_PARTIDA"].isna() | (check_transvar_p["C_PARTIDA"] == ""))
        & (check_transvar_p["C_CAPITULO"].isna() | (check_transvar_p["C_CAPITULO"] == ""))
        & (check_transvar_p["C_PROGRAMA"].isna() | (check_transvar_p["C_PROGRAMA"] == ""))
        & (check_transvar_p["C_ANALITICO"].isna() | (check_transvar_p["C_ANALITICO"] == ""))
    )
].copy()

# ALTER TABLE ... MODIFY C_ACTIVO CHAR(6): se refleja como truncado/normalizado a 6 caracteres
check_transvar_p["C_ACTIVO"] = check_transvar_p["C_ACTIVO"].astype(str).str.slice(0, 6)


In [ ]:
# Limpieza y reglas de negocio sobre OBS (UPDATEs condicionales del SAS)
check_transvar_p.loc[check_transvar_p["OBS"] == "Ingresar texto si aplica", "OBS"] = ""

check_transvar_p.loc[
    (~check_transvar_p["C_SCN"].isin(["D63-P2", "D63-P", "P3", "P2"]))
    & (check_transvar_p["OBS"] != ""),
    "OBS",
] = ""

mask_d63 = check_transvar_p["C_SCN"].isin(["D63-P2", "D63-P"]) & (check_transvar_p["OBS"] == "")
check_transvar_p.loc[mask_d63, "OBS"] = "CR18_difcoy"
check_transvar_p.loc[mask_d63, "C_ACTIVO"] = "D63"

mask_p3 = (check_transvar_p["C_SCN"] == "P3") & (check_transvar_p["OBS"] == "")
check_transvar_p.loc[mask_p3, "OBS"] = "CR18_difact"

mask_p2 = (
    (check_transvar_p["C_SCN"] == "P2")
    & (check_transvar_p["C_CUP"].isin(["EEPP", "SSPP"]))
    & (check_transvar_p["OBS"] == "")
)
check_transvar_p.loc[mask_p2, "OBS"] = "CR18_difact"
_log("check_transvar_p", check_transvar_p)


In [ ]:
# Anexa clasificaciones faltantes: DELETE ... WHERE EXISTS es un semi-join booleano por fila (nunca duplica).
# Se extrae SOLO las claves de la BD (mínimo necesario) y se deduplican ANTES del anti-join
# para evitar que un merge con claves repetidas en GOBGENER.CR18_T_SCN_2 (ver CHECK_ANALIT_VAR)
# infle el número de filas de check_transvar_p vía producto cartesiano.
claves_cols = [
    "C_PARTIDA", "C_CAPITULO", "C_PROGRAMA", "T_CUENTA", "C_CUENTA",
    "C_ITEM", "C_ASIGNACION", "C_ANALITICO", "ENTIDAD",
]
sql_claves_scn2 = """
SELECT DISTINCT C_PARTIDA, C_CAPITULO, C_PROGRAMA, T_CUENTA, C_CUENTA,
       C_ITEM, C_ASIGNACION, C_ANALITICO, ENTIDAD
FROM GOBGENER.dbo.CR18_T_SCN_2
"""
claves_scn2 = pd.read_sql(text(sql_claves_scn2), engine)
# drop_duplicates es redundante con el DISTINCT del SQL pero se deja explícito como salvaguarda
claves_scn2 = claves_scn2.drop_duplicates(subset=claves_cols)

merged = check_transvar_p.merge(claves_scn2, on=claves_cols, how="left", indicator=True)
check_transvar_p = check_transvar_p.loc[merged["_merge"].values == "left_only"].reset_index(drop=True)
_log("check_transvar_p_a_anexar", check_transvar_p)


In [ ]:
# PROC DATASETS APPEND: acumula sobre GOBGENER.CR18_T_SCN_2 (mismo comportamiento del SAS: re-ejecutar duplica)
check_transvar_p.to_sql("CR18_T_SCN_2", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND GOBGENER.dbo.CR18_T_SCN_2", len(check_transvar_p))


In [ ]:
# Verifica duplicados en la base de clasificaciones variables (recién anexada)
sql_check_analit_var = """
SELECT ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, T_CUENTA, C_CUENTA,
       C_ITEM, C_ASIGNACION, C_ANALITICO, COUNT(C_SCN) AS ID
FROM GOBGENER.dbo.CR18_T_SCN_2
GROUP BY ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, T_CUENTA, C_CUENTA,
         C_ITEM, C_ASIGNACION, C_ANALITICO
"""
check_analit_var = pd.read_sql(text(sql_check_analit_var), engine)
check_analit_var = check_analit_var[check_analit_var["ID"] != 1].reset_index(drop=True)
_log("check_analit_var", check_analit_var)


## 6a-Check Nvas TCons_mod

*confianza: medium · verificador: approve · SAS: PROC SQL CREATE TABLE con doble LEFT JOIN + DELETE/UPDATE post-proceso + PROC EXPORT EXCELCS*

In [ ]:
# ========= 6a-Check Nvas TCons_mod =========
sql_check_tcons = """
SELECT DISTINCT t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO AS ANALITICO,
       t1.C_ENTIDAD AS ENTIDAD,
       t2.Clasif_Tcons,
       t2.M_Muestra,
       t3.C_SCN,
       t3.C_CAGENTE,
       t1.C_PARTIDA,
       t1.N_PARTIDA,
       t1.C_CAPITULO,
       t1.N_CAPITULO,
       t1.C_PROGRAMA,
       t1.N_PROGRAMA,
       t3.FECHA,
       t3.ANALISTA
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.ANALIT_TCONS t2
    ON t1.C_ENTIDAD = t2.ENTIDAD AND t2.C_ASIGNACION = t1.C_ASIGNACION
   AND t2.C_ANALITICO = t1.C_ANALITICO AND t2.C_CUENTA = t1.C_CUENTA AND t2.C_ITEM = t1.C_ITEM
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t3
    ON t1.C_PARTIDA = t3.C_PARTIDA AND t1.C_CAPITULO = t3.C_CAPITULO
   AND t1.C_PROGRAMA = t3.C_PROGRAMA AND t1.C_ASIGNACION = t3.C_ASIGNACION
   AND t1.C_ANALITICO = t3.C_ANALITICO AND t1.C_CUENTA = t3.C_CUENTA AND t1.C_ITEM = t3.C_ITEM
   AND t1.C_ENTIDAD = t3.Entidad
WHERE t2.Clasif_Tcons IS NULL
  AND t1.AÑO = :year
  AND t1.C_CUENTA IN ('24','33')
  AND t1.C_ITEM IN ('01','03','06','09','08')
"""
check_tcons = pd.read_sql(text(sql_check_tcons), engine, params={"year": Year})
_log("check_tcons", check_tcons)


In [ ]:
# DELETE FROM CHECK_TCONS WHERE C_ITEM='08' AND C_SCN<>'TC' (filtro negativo del SAS original)
check_tcons = check_tcons[~((check_tcons["C_ITEM"] == "08") & (check_tcons["C_SCN"] != "TC"))]
# UPDATE CHECK_TCONS SET Clasif_Tcons='OK' WHERE C_SCN NOT IN ('TC')
check_tcons.loc[check_tcons["C_SCN"] != "TC", "Clasif_Tcons"] = "OK"
_log("check_tcons", check_tcons)


In [ ]:
# PROC EXPORT DBMS=EXCELCS: bridge PC Files Server -> se lee/escribe directo por la ruta del archivo (M-001: ruta parametrizada, relativa al workspace)
ruta_check_tcons = Path("salidas") / "CHECK_TCONS.xlsb"
check_tcons.to_excel(ruta_check_tcons, sheet_name="TCONS", index=False)


## 6b-Clasif Nvas TCons_mod

*confianza: medium · verificador: approve · SAS: PROC IMPORT EXCELCS + PROC SQL UPDATE/DELETE + PROC DATASETS APPEND + LEFT JOIN de verificación*

In [ ]:
# ========= 6b-Clasif Nvas TCons_mod =========
# M-001: ruta hardcodeada consolidada en variable parametrizada
ruta_check_tcons = Path("datos") / "GobiernoGeneral" / "CHECK_TCONS.xlsb"
check_tcons_p = pd.read_excel(ruta_check_tcons, sheet_name="TCONS", engine="pyxlsb")


In [ ]:
check_tcons_p["CLASIF_TCONS"] = check_tcons_p["CLASIF_TCONS"].str.upper()
check_tcons_p.loc[check_tcons_p["CLASIF_TCONS"].isin(["M", "MM", "MGCG", "MGG"]), "M_MUESTRA"] = "0"
_log("check_tcons_p", check_tcons_p)


In [ ]:
check_tcons_r = check_tcons_p[
    (~check_tcons_p["CLASIF_TCONS"].isin(["M", "MM", "MGCG", "MGG", "OK"]))
    & (check_tcons_p["C_CUENTA"].notna())
    & (check_tcons_p["C_CUENTA"] != "")
]
_log("check_tcons_r", check_tcons_r)


In [ ]:
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM GOBGENER.dbo.ANALIT_TCONS WHERE CLASIF_TCONS = :vacio"), {"vacio": ""})
    _log("DELETE GOBGENER.dbo.ANALIT_TCONS (sin clasificacion)", res.rowcount)


In [ ]:
analit_tcons_actual = pd.read_sql(
    text("SELECT C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, ENTIDAD FROM GOBGENER.dbo.ANALIT_TCONS"),
    engine,
)
claves = ["C_CUENTA", "C_ITEM", "C_ASIGNACION", "C_ANALITICO", "ENTIDAD"]
merged = check_tcons_p.merge(analit_tcons_actual, on=claves, how="left", indicator=True)
check_tcons_p = merged[merged["_merge"] == "left_only"].drop(columns=["_merge"])
_log("check_tcons_p_filtrado", check_tcons_p)


In [ ]:
# PROC APPEND: acumula sobre GOBGENER.ANALIT_TCONS, re-ejecutar duplica igual que el SAS original
check_tcons_p.to_sql("ANALIT_TCONS", engine, schema="dbo", if_exists="append", index=False)


In [ ]:
sql_check_tcons_falt = """
SELECT T1.ENTIDAD, T1.C_CUENTA, T1.C_ITEM, T1.C_ANALITICO, T1.ANALITICO, T1.C_SCN,
       T1.C_CAGENTE, T1.FECHA, T1.ANALISTA, T2.CLASIF_TCONS
FROM GOBGENER.dbo.CR18_T_SCN_2 T1
LEFT JOIN GOBGENER.dbo.ANALIT_TCONS T2
    ON T1.ENTIDAD = T2.ENTIDAD AND T1.C_CUENTA = T2.C_CUENTA
    AND T1.C_ITEM = T2.C_ITEM AND T1.C_ANALITICO = T2.C_ANALITICO
WHERE T1.C_SCN = :c_scn AND (T2.CLASIF_TCONS = :vacio OR T2.CLASIF_TCONS IS NULL)
ORDER BY T1.FECHA
"""
check_tcons_falt = pd.read_sql(text(sql_check_tcons_falt), engine, params={"c_scn": "TC", "vacio": ""})
_log("check_tcons_falt", check_tcons_falt)


## 7a-Check Est_Exp_MM TCons_mod

*confianza: medium · verificador: approve · SAS: PROC SQL LEFT JOIN + DATA step expansion + PROC EXPORT/IMPORT EXCELCS + PROC SQL DELETE/GROUP BY + PROC DATASETS APPEND*

In [ ]:
# ========= 7a-Check Est_Exp_MM TCons_mod =========
sql_check = """
SELECT DISTINCT t1.C_CUENTA, t1.C_ITEM, t1.ENTIDAD, t1.C_ASIGNACION, t1.C_ANALITICO, t1.ANALITICO,
       '' AS C_SCN, CAST(NULL AS float) AS POND, '' AS C_CAGENTE
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN GOBGENER.dbo.EST_EXP_MM t2
  ON t1.ENTIDAD = t2.ENTIDAD AND t2.C_ASIGNACION = t1.C_ASIGNACION
 AND t2.C_ANALITICO = t1.C_ANALITICO AND t2.C_CUENTA = t1.C_CUENTA AND t2.C_ITEM = t1.C_ITEM
WHERE t1.Clasif_Tcons = 'MM' AND t1.M_Muestra = 0 AND t2.ENTIDAD IS NULL AND t2.C_ANALITICO IS NULL
"""
check_est_exp_mm = pd.read_sql(text(sql_check), engine)
_log("check_est_exp_mm", check_est_exp_mm)


In [ ]:
# genera N_FALTANTE (SAS: MOD(1,_N_)+1)
check_est_exp_mm = check_est_exp_mm.reset_index(drop=True)
n_ = check_est_exp_mm.index.to_series() + 1
check_est_exp_mm["N_FALTANTE"] = (1 % n_) + 1

# repite cada fila 12 veces (DO I=1 TO 12; OUTPUT;) preservando el orden original
check_est_exp_mm = check_est_exp_mm.loc[check_est_exp_mm.index.repeat(12)].reset_index(drop=True)

# genera MES = MOD(_N_-1,12)+1 sobre la fila expandida
n_expandida = check_est_exp_mm.index.to_series() + 1
check_est_exp_mm["MES"] = ((n_expandida - 1) % 12) + 1
_log("check_est_exp_mm", check_est_exp_mm)


In [ ]:
# PROC EXPORT DBMS=EXCELCS: se escribe directo por la ruta UNC del DATAFILE, sin bridge/servidor intermedio
# M-001: ruta hardcodeada del SAS reemplazada por ruta relativa al workspace
ruta_check_est_exp_mm = pathlib.Path("salidas") / "Clasificaciones" / "CHECK_EST_EXP_MM.xlsb"
ruta_check_est_exp_mm.parent.mkdir(parents=True, exist_ok=True)
check_est_exp_mm.to_excel(ruta_check_est_exp_mm, sheet_name="CHECK_EST_EXP_MM", index=False)


In [ ]:
# módulo 2: importa las clasificaciones llenadas manualmente en el Excel exportado arriba
# M-001: ruta hardcodeada del SAS reemplazada por ruta relativa al workspace (mismo archivo que el export)
ruta_est_exp_mm_c = pathlib.Path("salidas") / "Clasificaciones" / "CHECK_EST_EXP_MM.xlsb"
est_exp_mm_c = pd.read_excel(ruta_est_exp_mm_c, sheet_name="CHECK_EST_EXP_MM", engine="pyxlsb")

# PROC SQL; DELETE FROM EST_EXP_MM_C WHERE C_CUENTA="" AND C_ITEM="";
est_exp_mm_c = est_exp_mm_c[~((est_exp_mm_c["C_CUENTA"] == "") & (est_exp_mm_c["C_ITEM"] == ""))]
_log("est_exp_mm_c", est_exp_mm_c)


In [ ]:
# CHECK_MM_TOTAL_C: verifica que las ponderaciones sumen 1 por grupo (queda con los que NO cuadran)
check_mm_total_c = (
    est_exp_mm_c.groupby(["MES", "C_CUENTA", "C_ITEM", "ENTIDAD", "C_ASIGNACION", "C_ANALITICO"], dropna=False)["POND"]
    .sum()
    .reset_index(name="TOTAL")
)
check_mm_total_c = check_mm_total_c[check_mm_total_c["TOTAL"] != 1]
_log("check_mm_total_c", check_mm_total_c)


In [ ]:
# CHECK_MM_ANALIT: chequea consistencia del C_ANALITICO contra C_ITEM+C_ASIGNACION; vacío = OK
analit_concat = est_exp_mm_c["C_ITEM"].astype(str) + est_exp_mm_c["C_ASIGNACION"].astype(str)
prefijo_analitico = est_exp_mm_c["C_ANALITICO"].astype(str).str[:-3]
mask_rev = prefijo_analitico != analit_concat

check_mm_analit = est_exp_mm_c.loc[mask_rev].copy()
check_mm_analit["ANALIT"] = analit_concat.loc[mask_rev]
check_mm_analit["ID"] = "REV"
check_mm_analit = (
    check_mm_analit.groupby(["MES", "C_CUENTA", "C_ITEM", "ENTIDAD", "C_ANALITICO", "ANALIT", "ID"], dropna=False)["POND"]
    .sum()
    .reset_index(name="TOTAL")
)
_log("check_mm_analit", check_mm_analit)


In [ ]:
# DATA EST_EXP_MM_C; SET EST_EXP_MM_C; DROP ANALITICO N_FALTANTE;
est_exp_mm_c = est_exp_mm_c.drop(columns=["ANALITICO", "N_FALTANTE"], errors="ignore")


In [ ]:
# PROC SQL DELETE ... WHERE EXISTS: se descartan de EST_EXP_MM_C las filas cuya clave ya existe en la tabla real
claves_existentes = pd.read_sql(
    text("SELECT DISTINCT C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, ENTIDAD FROM GOBGENER.dbo.EST_EXP_MM"),
    engine,
)
claves_existentes["_existe"] = True
est_exp_mm_c = est_exp_mm_c.merge(
    claves_existentes, on=["C_CUENTA", "C_ITEM", "C_ASIGNACION", "C_ANALITICO", "ENTIDAD"], how="left"
)
est_exp_mm_c = est_exp_mm_c[est_exp_mm_c["_existe"].isna()].drop(columns=["_existe"])
_log("est_exp_mm_c", est_exp_mm_c)


In [ ]:
# PROC DATASETS; APPEND BASE=GOBGENER.EST_EXP_MM DATA=EST_EXP_MM_C FORCE; -> SAS acumulaba, se traduce como append
est_exp_mm_c.to_sql("EST_EXP_MM", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND GOBGENER.dbo.EST_EXP_MM", len(est_exp_mm_c))


## 7b-Check Est_Exp_M TCons_mod

*confianza: medium · verificador: approve · SAS: PROC SQL LEFT JOIN + export/import Excel (bridge PC Files Server) + PROC SQL UPDATE/DELETE + PROC DATASETS APPEND*

In [ ]:
# ========= 7b-Check Est_Exp_M TCons_mod =========
sql_check = """
SELECT DISTINCT t1.C_CUENTA, t1.C_ITEM, t1.ENTIDAD, t1.C_ASIGNACION, t1.C_ANALITICO, t1.ANALITICO,
       '' AS C_SCN, CAST(NULL AS float) AS POND, '' AS C_CAGENTE, '' AS C_CUP
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN GOBGENER.dbo.EST_EXP_MIXTAS_M t2
  ON t1.ENTIDAD = t2.ENTIDAD AND t2.C_ASIGNACION = t1.C_ASIGNACION
 AND t2.C_ANALITICO = t1.C_ANALITICO AND t2.C_CUENTA = t1.C_CUENTA AND t2.C_ITEM = t1.C_ITEM
WHERE t1.Clasif_Tcons = 'M' AND t1.M_Muestra = 0 AND t2.ENTIDAD IS NULL AND t2.C_ANALITICO IS NULL
"""
check_est_exp_m = pd.read_sql(text(sql_check), engine)
_log("check_est_exp_m", check_est_exp_m)


In [ ]:
# genera N_FALTANTE (numero de fila) y repite cada fila 12 veces para los meses del ano
check_est_exp_m = check_est_exp_m.reset_index(drop=True)
check_est_exp_m["N_FALTANTE"] = check_est_exp_m.index + 1
check_est_exp_m = check_est_exp_m.loc[check_est_exp_m.index.repeat(12)].reset_index(drop=True)
check_est_exp_m["MES"] = (check_est_exp_m.index % 12) + 1
_log("check_est_exp_m", check_est_exp_m)


In [ ]:
# M-001: ruta hardcodeada -> relativa al workspace
ruta_check_est_exp_m = Path("salidas") / "Clasificaciones" / "CHECK_EST_EXP_M.xlsb"
ruta_check_est_exp_m.parent.mkdir(parents=True, exist_ok=True)
check_est_exp_m.to_excel(ruta_check_est_exp_m, sheet_name="CHECK_EST_EXP_M", index=False)


In [ ]:
# modulo 2: reimporta el excel ya clasificado manualmente (mismo archivo exportado arriba)
# el bridge PC Files Server (EXCELCS/SERVER_NAME) de SAS se traduce a lectura directa del archivo
est_exp_m_c = pd.read_excel(ruta_check_est_exp_m, sheet_name="CHECK_EST_EXP_M", engine="pyxlsb")
est_exp_m_c = est_exp_m_c[~((est_exp_m_c["C_CUENTA"] == "") & (est_exp_m_c["C_ITEM"] == ""))]
_log("est_exp_m_c", est_exp_m_c)


In [ ]:
# verifica que las estructuras ingresadas sumen 1 (POND) por grupo
check_mixtas_tot = (
    est_exp_m_c.groupby(["MES", "C_CUENTA", "C_ITEM", "ENTIDAD", "C_ASIGNACION", "C_ANALITICO"], dropna=False)["POND"]
    .sum()
    .reset_index(name="TOTAL")
)
check_mixtas_tot["TOTAL2"] = check_mixtas_tot["TOTAL"].round(1)
check_mixtas_tot = check_mixtas_tot[check_mixtas_tot["TOTAL2"].round(0) != 1]
_log("check_mixtas_tot", check_mixtas_tot)


In [ ]:
# chequea consistencia de C_ANALITICO vs (C_ITEM + C_ASIGNACION): si vacia, todo OK
analit_calc = est_exp_m_c["C_ITEM"].astype(str) + est_exp_m_c["C_ASIGNACION"].astype(str)
id_calc = np.where(
    est_exp_m_c["C_ANALITICO"].astype(str).str[:-3] != analit_calc,
    "REV",
    "OK",
)
tmp = est_exp_m_c.assign(ANALIT=analit_calc, ID=id_calc)
tmp = tmp[tmp["C_ANALITICO"].astype(str).str[:-3] != analit_calc]
check_mixtas_analit = (
    tmp.groupby(["C_CUENTA", "C_ITEM", "ENTIDAD", "C_ANALITICO", "ANALIT", "ID"], dropna=False)["POND"]
    .sum()
    .reset_index(name="TOTAL")
)
_log("check_mixtas_analit", check_mixtas_analit)


In [ ]:
# excepciones manuales: cruces malos conocidos que no generan problema (marcados OK2 y descartados)
excepciones = [
    ("03027000", "01027000", 1301),
    ("03515000", "01515000", 6104),
    ("03604000", "01604000", 6104),
    ("01114000", "01004000", 24404),
    ("09010000", "09007000", 26706),
]
mask_ok2 = pd.Series(False, index=check_mixtas_analit.index)
for c_analitico, analit, entidad in excepciones:
    mask_ok2 |= (
        (check_mixtas_analit["C_ANALITICO"] == c_analitico)
        & (check_mixtas_analit["ANALIT"] == analit)
        & (check_mixtas_analit["ENTIDAD"] == entidad)
    )
check_mixtas_analit = check_mixtas_analit[~mask_ok2]
_log("check_mixtas_analit", check_mixtas_analit)


In [ ]:
# anexa datos a la tabla principal: descarta columnas auxiliares y filas ya existentes en destino
est_exp_m_c = est_exp_m_c.drop(columns=["ANALITICO", "N_FALTANTE"], errors="ignore")

existentes = pd.read_sql(
    text("SELECT C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, ENTIDAD FROM GOBGENER.dbo.EST_EXP_MIXTAS_M"),
    engine,
)
claves = ["C_CUENTA", "C_ITEM", "C_ASIGNACION", "C_ANALITICO", "ENTIDAD"]
merged = est_exp_m_c.merge(existentes[claves].drop_duplicates(), on=claves, how="left", indicator=True)
est_exp_m_c = merged[merged["_merge"] == "left_only"].drop(columns=["_merge"])
_log("est_exp_m_c", est_exp_m_c)


In [ ]:
# PROC APPEND: acumula tal cual, sin idempotencia (igual que el SAS original)
est_exp_m_c.to_sql("EST_EXP_MIXTAS_M", engine, schema="dbo", if_exists="append", index=False)


## 7c-Check Est_MM TCons

*confianza: high · verificador: approve · SAS: PROC SQL CREATE TABLE con LEFT JOIN y filtro de no-coincidencia (antijoin)*

In [ ]:
# ========= 7c-Check Est_MM TCons =========
sql_check_est_mm = """
SELECT DISTINCT t1.*
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN GOBGENER.dbo.EST_MM t2
    ON t1.ENTIDAD = t2.ENTIDAD
    AND t2.C_ASIGNACION = t1.C_ASIGNACION
    AND t2.C_ANALITICO = t1.C_ANALITICO
    AND t2.C_CUENTA = t1.C_CUENTA
    AND t2.C_ITEM = t1.C_ITEM
WHERE t1.Clasif_Tcons = 'MM' AND t1.M_Muestra = 1 AND t2.ENTIDAD IS NULL AND t2.C_ANALITICO IS NULL
"""
check_est_mm = pd.read_sql(text(sql_check_est_mm), engine)
_log("check_est_mm", check_est_mm)


## 7d-Check Est_M TCons

*confianza: medium · verificador: approve · SAS: PROC SQL CREATE TABLE con LEFT JOIN y filtro de no-match (antijoin)*

In [ ]:
# ========= 7d-Check Est_M TCons =========
sql_check_est_m = """
SELECT DISTINCT t1.*
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN GOBGENER.dbo.EST_MIXTAS_M t2
    ON t1.ENTIDAD = t2.ENTIDAD
    AND t2.C_ASIGNACION = t1.C_ASIGNACION
    AND t2.C_ANALITICO = t1.C_ANALITICO
    AND t2.C_CUENTA = t1.C_CUENTA
    AND t2.C_ITEM = t1.C_ITEM
WHERE t1.Clasif_Tcons = 'M' AND t1.M_Muestra = 1
    AND t2.ENTIDAD IS NULL AND t2.C_ANALITICO IS NULL
"""
check_est_m = pd.read_sql(text(sql_check_est_m), engine)
_log("check_est_m", check_est_m)


## 7e-Check Est_Exp_MGCG TCons_mod

*confianza: medium · verificador: approve · SAS: PROC SQL LEFT JOIN (detección de faltantes) + PROC SQL LEFT JOIN por C_CUENTA con filtro C_ITEM='03' + PROC EXPORT/IMPORT XLSB + PROC SQL DELETE + PROC DATASETS APPEND*

In [ ]:
# ========= 7e-Check Est_Exp_MGCG TCons_mod =========
sql_check_est_exp_mgcg = """
SELECT t1.C_CUENTA, t1.C_ITEM, COUNT(t1.C_ASIGNACION) AS ID
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN GOBGENER.dbo.EST_EXP_MGCG t2
  ON t2.C_CUENTA = t1.C_CUENTA AND t2.C_ITEM = t1.C_ITEM
WHERE t1.Clasif_Tcons = 'MGCG' AND t1.M_Muestra = 0 AND t2.C_SCN IS NULL
GROUP BY t1.C_CUENTA, t1.C_ITEM
"""
check_est_exp_mgcg = pd.read_sql(text(sql_check_est_exp_mgcg), engine)
check_est_exp_mgcg = check_est_exp_mgcg.drop_duplicates()
_log("check_est_exp_mgcg", check_est_exp_mgcg)


In [ ]:
sql_check_est_exp_mgcg_p = """
SELECT t1.C_CUENTA, t1.C_ITEM, t3.C_SCN, t3.POND, t3.C_CAGENTE, t3.MES
FROM (
    SELECT DISTINCT t1a.C_CUENTA, t1a.C_ITEM
    FROM GOBGENER.dbo.ANALIT_TCONS t1a
    LEFT JOIN GOBGENER.dbo.EST_EXP_MGCG t2a
      ON t2a.C_CUENTA = t1a.C_CUENTA AND t2a.C_ITEM = t1a.C_ITEM
    WHERE t1a.Clasif_Tcons = 'MGCG' AND t1a.M_Muestra = 0 AND t2a.C_SCN IS NULL
) t1
LEFT JOIN GOBGENER.dbo.EST_EXP_MGCG t3
  ON t3.C_CUENTA = t1.C_CUENTA AND t3.C_ITEM = '03'
"""
# t1 replica exactamente el universo de CHECK_EST_EXP_MGCG (cuentas/item SIN estructura);
# el LEFT JOIN final se hace SOLO por C_CUENTA contra EST_EXP_MGCG filtrado a C_ITEM='03',
# preservando filas de t1 aunque no exista item '03' para esa cuenta (igual que el SAS original)
check_est_exp_mgcg_p = pd.read_sql(text(sql_check_est_exp_mgcg_p), engine)
_log("check_est_exp_mgcg_p", check_est_exp_mgcg_p)


In [ ]:
ruta_check_est_exp_mgcg_p = Path("salidas") / "Clasificaciones" / "CHECK_EST_EXP_MGCG_P.xlsb"
check_est_exp_mgcg_p.to_excel(ruta_check_est_exp_mgcg_p, sheet_name="PROPUESTA", index=False)


In [ ]:
ruta_import_check_est_exp_mgcg_p = Path("salidas") / "Clasificaciones" / "CHECK_EST_EXP_MGCG_P.xlsb"
est_exp_mgcg_p = pd.read_excel(ruta_import_check_est_exp_mgcg_p, sheet_name="PROPUESTA", engine="pyxlsb")
est_exp_mgcg_p = est_exp_mgcg_p[~((est_exp_mgcg_p["C_CUENTA"] == "") & (est_exp_mgcg_p["C_ITEM"] == ""))]
_log("est_exp_mgcg_p", est_exp_mgcg_p)


In [ ]:
check_mgcg_tot = (
    est_exp_mgcg_p.groupby(["MES", "C_CUENTA", "C_ITEM"], as_index=False)["POND"]
    .sum()
    .rename(columns={"POND": "TOTAL"})
)
check_mgcg_tot["TOTAL2"] = check_mgcg_tot["TOTAL"].round(1)
check_mgcg_tot = check_mgcg_tot[check_mgcg_tot["TOTAL2"] != 1]
_log("check_mgcg_tot", check_mgcg_tot)


In [ ]:
estructura_actual = pd.read_sql(text("SELECT C_CUENTA, C_ITEM FROM GOBGENER.dbo.EST_EXP_MGCG"), engine)
claves_existentes = set(zip(estructura_actual["C_CUENTA"], estructura_actual["C_ITEM"]))
est_exp_mgcg_p_new = est_exp_mgcg_p[
    ~est_exp_mgcg_p.apply(lambda r: (r["C_CUENTA"], r["C_ITEM"]) in claves_existentes, axis=1)
]
_log("est_exp_mgcg_p_new", est_exp_mgcg_p_new)


In [ ]:
est_exp_mgcg_p_new.to_sql("EST_EXP_MGCG", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND GOBGENER.dbo.EST_EXP_MGCG", len(est_exp_mgcg_p_new))


## 7f-Check Est_Exp_MGG TCons_mod

*confianza: low · verificador: approve · SAS: PROC SQL CREATE TABLE + PROC EXPORT/IMPORT (bridge PC Files EXCELCS) + PROC APPEND*

In [ ]:
# ========= 7f-Check Est_Exp_MGG TCons_mod =========
sql_check = """
SELECT DISTINCT t1.C_CUENTA, t1.C_ITEM, COUNT(t1.C_ASIGNACION) AS ID
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN GOBGENER.dbo.EST_EXP_MGG t2
    ON t2.C_CUENTA = t1.C_CUENTA AND t2.C_ITEM = t1.C_ITEM
WHERE t1.Clasif_Tcons = 'MGG' AND t1.M_Muestra = 0 AND t2.C_SCN IS NULL
GROUP BY t1.C_CUENTA, t1.C_ITEM
"""
check_est_exp_mgg = pd.read_sql(text(sql_check), engine)
_log("check_est_exp_mgg", check_est_exp_mgg)


In [ ]:
if check_est_exp_mgg.empty:
    check_est_exp_mgg_p = pd.DataFrame(columns=["C_CUENTA", "C_ITEM", "C_SCN", "POND", "C_CAGENTE", "MES"])
else:
    cuentas = tuple(check_est_exp_mgg["C_CUENTA"].unique().tolist())
    sql_est_mgg = """
    SELECT C_CUENTA, C_ITEM, C_SCN, POND, C_CAGENTE, MES
    FROM GOBGENER.dbo.EST_EXP_MGG
    WHERE C_ITEM = '01' AND C_CUENTA IN :cuentas
    """
    est_exp_mgg_filtro = pd.read_sql(text(sql_est_mgg), engine, params={"cuentas": cuentas})
    # réplica del JOIN + GROUP BY SUM(POND) del SAS: t1 aporta C_CUENTA/C_ITEM, t2 aporta el resto
    merged = check_est_exp_mgg[["C_CUENTA", "C_ITEM"]].merge(
        est_exp_mgg_filtro, on="C_CUENTA", how="left", suffixes=("", "_t2")
    )
    check_est_exp_mgg_p = (
        merged.groupby(["C_CUENTA", "C_ITEM", "C_SCN", "C_CAGENTE", "MES"], dropna=False)["POND"]
        .sum()
        .reset_index()
    )
    check_est_exp_mgg_p = check_est_exp_mgg_p[["C_CUENTA", "C_ITEM", "C_SCN", "POND", "C_CAGENTE", "MES"]]
_log("check_est_exp_mgg_p", check_est_exp_mgg_p)


In [ ]:
# M-001: ruta hardcodeada del SAS reemplazada por ruta relativa al workspace
ruta_check_mgg_p = Path("salidas") / "clasificaciones" / "CHECK_EST_EXP_MGG_P.xlsb"
ruta_check_mgg_p.parent.mkdir(parents=True, exist_ok=True)
check_est_exp_mgg_p.to_excel(ruta_check_mgg_p, sheet_name="PROPUESTA", index=False, engine="openpyxl")


In [ ]:
# Reimporta la propuesta exportada (equivalente al PROC IMPORT posterior sobre el mismo archivo)
est_exp_mgg_p = pd.read_excel(ruta_check_mgg_p, sheet_name="PROPUESTA", engine="pyxlsb")
est_exp_mgg_p = est_exp_mgg_p[~((est_exp_mgg_p["C_CUENTA"].isna() | (est_exp_mgg_p["C_CUENTA"] == "")) & (est_exp_mgg_p["C_ITEM"].isna() | (est_exp_mgg_p["C_ITEM"] == "")))]
_log("est_exp_mgg_p", est_exp_mgg_p)


In [ ]:
check_mgg_tot = (
    est_exp_mgg_p.groupby(["MES", "C_CUENTA", "C_ITEM"], dropna=False)["POND"]
    .sum()
    .reset_index()
    .rename(columns={"POND": "TOTAL"})
)
check_mgg_tot["TOTAL2"] = check_mgg_tot["TOTAL"].round(1)
check_mgg_tot = check_mgg_tot[check_mgg_tot["TOTAL2"] != 1]
_log("check_mgg_tot", check_mgg_tot)


In [ ]:
# Excluye de la propuesta las combinaciones C_CUENTA/C_ITEM que ya existen en la tabla principal
existentes = pd.read_sql(text("SELECT DISTINCT C_CUENTA, C_ITEM FROM GOBGENER.dbo.EST_EXP_MGG"), engine)
est_exp_mgg_p = est_exp_mgg_p.merge(
    existentes.assign(_existe=True), on=["C_CUENTA", "C_ITEM"], how="left"
)
est_exp_mgg_p = est_exp_mgg_p[est_exp_mgg_p["_existe"].isna()].drop(columns=["_existe"])
_log("est_exp_mgg_p_final", est_exp_mgg_p)


In [ ]:
# PROC APPEND: acumula tal cual el SAS original (re-ejecutar duplica, igual que el SAS)
est_exp_mgg_p.to_sql("EST_EXP_MGG", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND GOBGENER.dbo.EST_EXP_MGG", len(est_exp_mgg_p))


## 9a-Check Nvas Ent COFOG

*confianza: medium · verificador: approve · SAS: PROC SQL CREATE TABLE con LEFT JOIN (condición ON con tautología t1.C_PROGRAMA=t1.C_PROGRAMA) y filtro WHERE IS NULL*

In [ ]:
# ========= 9a-Check Nvas Ent COFOG =========
sql_ejecucion = """
SELECT C_ENTIDAD, C_PARTIDA, N_PARTIDA, C_CAPITULO, N_CAPITULO, C_PROGRAMA, N_PROGRAMA
FROM GOBGENER.dbo.EJECUCION
WHERE AÑO = :year
"""
ejecucion_y = pd.read_sql(text(sql_ejecucion), engine, params={"year": Year})

cofog = pd.read_sql("SELECT ENTIDAD, C_PARTIDA, C_CAPITULO, C_COFOG, N_COFOG, C_COFOG_1, N_COFOG_1 FROM GOBGENER.dbo.COFOG", engine)


In [ ]:
check_cofog_entid = ejecucion_y.merge(
    cofog,
    left_on=["C_ENTIDAD", "C_PARTIDA", "C_CAPITULO"],
    right_on=["ENTIDAD", "C_PARTIDA", "C_CAPITULO"],
    how="left",
    suffixes=("", "_cofog"),
)
check_cofog_entid = check_cofog_entid[check_cofog_entid["C_COFOG"].isna()]
check_cofog_entid = check_cofog_entid.rename(columns={"C_ENTIDAD": "ENTIDAD"})
check_cofog_entid = check_cofog_entid[
    ["ENTIDAD", "C_COFOG", "N_COFOG", "C_COFOG_1", "N_COFOG_1", "C_PARTIDA", "N_PARTIDA", "C_CAPITULO", "N_CAPITULO", "C_PROGRAMA", "N_PROGRAMA"]
].drop_duplicates()
_log("check_cofog_entid", check_cofog_entid)


## 9b-Clasif COFOG de Nvas Ent

*confianza: low · verificador: approve · SAS: PROC DATASETS; APPEND BASE=... DATA=... FORCE;*

In [ ]:
# ========= 9b-Clasif COFOG de Nvas Ent =========
check_cofog_entid = pd.read_sql(text("SELECT * FROM WORK.CHECK_COFOG_ENTID"), engine) if False else check_cofog_entid


In [ ]:
if check_cofog_entid.empty:
    raise NotImplementedError("CHECK_COFOG_ENTID llega vacío desde CodeTask-EKp3vGrePAlPMEBt (9a-Check Nvas Ent COFOG); el PROC APPEND de SAS con FORCE no valida esto y anexaría 0 filas en silencio")

check_cofog_entid.to_sql("COFOG", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND GOBGENER.dbo.COFOG", len(check_cofog_entid))


## 10a-Check Analit COFOG

*confianza: medium · verificador: approve · SAS: PROC SQL CREATE TABLE con múltiples LEFT JOIN y filtro WHERE*

In [ ]:
# ========= 10a-Check Analit COFOG =========
sql_check_cofog_analit = """
SELECT DISTINCT t1.SUBSECTOR,
                t1.MONEDA,
                t1.C_TIPO_CUENTA,
                t1.C_CUENTA,
                t1.C_ITEM,
                t2.C_SCN,
                t1.C_ANALITICO,
                t1.N_ANALITICO,
                t2.C_CAGENTE,
                t3.C_COFOG_1,
                t3.N_COFOG_1,
                t1.C_PARTIDA,
                t1.N_PARTIDA,
                t1.C_CAPITULO,
                t1.N_CAPITULO,
                t1.C_PROGRAMA,
                t1.N_PROGRAMA
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.COFOG_T_SCN_2 t3
    ON t1.C_PARTIDA = t3.C_PARTIDA AND t1.C_CAPITULO = t3.C_CAPITULO
   AND t1.C_PROGRAMA = t3.C_PROGRAMA AND t1.C_CUENTA = t3.C_CUENTA
   AND t1.C_ANALITICO = t3.C_ANALITICO AND t1.MONEDA = t3.MONEDA
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO
   AND t1.C_PROGRAMA = t2.C_PROGRAMA AND t1.C_TIPO_CUENTA = t2.T_CUENTA
   AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
   AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
    ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t3.C_COFOG_1 IS NULL
  AND t1.AÑO = :anio
  AND t1.C_CUENTA IN ('24', '33')
  AND t4.NIVEL = 'REGIONAL'
"""
check_cofog_analit = pd.read_sql(text(sql_check_cofog_analit), engine, params={"anio": Year})
_log("check_cofog_analit", check_cofog_analit)


## 10b-Clasif Analit COFOG

*confianza: low · verificador: approve · SAS: DATA step RENAME + PROC DATASETS APPEND FORCE*

In [ ]:
# ========= 10b-Clasif Analit COFOG =========
check_cofog_analit = check_cofog_analit.rename(columns={"N_ANALITICO": "ANALITICO"})
_log("check_cofog_analit", check_cofog_analit)


In [ ]:
check_cofog_analit.to_sql("COFOG_T_SCN_2", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND GOBGENER.dbo.COFOG_T_SCN_2", len(check_cofog_analit))


## 8a-Check Nvas Ent ACTIV

*confianza: medium · verificador: approve · SAS: PROC SQL CREATE TABLE (LEFT JOIN + WHERE IS NULL) seguido de 5 PROC SQL UPDATE condicionales encadenados*

In [ ]:
# ========= 8a-Check Nvas Ent ACTIV =========
sql_check_activ = """
SELECT DISTINCT t1.C_ENTIDAD AS ENTIDAD,
       t2.ACTIVIDAD,
       t2.NIVEL,
       t1.C_PARTIDA,
       t1.N_PARTIDA,
       t1.C_CAPITULO,
       t1.N_CAPITULO,
       t1.C_PROGRAMA,
       t1.N_PROGRAMA
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t2 ON t1.C_ENTIDAD = t2.ENTIDAD
WHERE t2.ENTIDAD IS NULL AND t1.AÑO = :anio
"""
check_activ_entid = pd.read_sql(text(sql_check_activ), engine, params={"anio": Year})
_log("check_activ_entid", check_activ_entid)


In [ ]:
# Imputación de ACTIVIDAD/NIVEL según diccionario de partidas/capítulos (equivalente a los UPDATE encadenados del SAS)
# ACTIVIDAD/NIVEL inician vacíos como en el LEFT JOIN original (t2.ACTIVIDAD/NIVEL son NULL para las filas seleccionadas)
check_activ_entid["ACTIVIDAD"] = check_activ_entid["ACTIVIDAD"].fillna("")
check_activ_entid["NIVEL"] = check_activ_entid["NIVEL"].fillna("")

partidas_aapp_central = ['01','02','03','04','06','07','10','12','13','14','17','18','19','20','21','22','23','24','25','26','27','28','29']
cap_educ = ['11','18','19','21','22','23','24','25','26','27','28','29','30','31','32','33','34','35','36','37','38','39','40','41','42','43','44','45','46','47','48','49','50','51','52','53','54','55','56','57','58','59','60','61','62','63','64','65','66']

# 1) Partidas que se clasifican completamente en AAPP-CENTRAL
mask1 = check_activ_entid["C_PARTIDA"].isin(partidas_aapp_central)
check_activ_entid.loc[mask1, "ACTIVIDAD"] = "AAPP"
check_activ_entid.loc[mask1, "NIVEL"] = "CENTRAL"

# 2) Excepto cuando capitulo es Casa de Moneda (partida 08, capitulo != 06)
mask2 = (check_activ_entid["C_PARTIDA"] == "08") & (check_activ_entid["C_CAPITULO"] != "06") & (check_activ_entid["ACTIVIDAD"] == "")
check_activ_entid.loc[mask2, "ACTIVIDAD"] = "AAPP"
check_activ_entid.loc[mask2, "NIVEL"] = "CENTRAL"

# 3) Ministerio del Trabajo, Credito Prendario -> EXCLUIDO
mask3 = (check_activ_entid["C_PARTIDA"] == "15") & (check_activ_entid["C_CAPITULO"] == "04") & (check_activ_entid["ACTIVIDAD"] == "")
check_activ_entid.loc[mask3, "ACTIVIDAD"] = "EXCLUIDO"
check_activ_entid.loc[mask3, "NIVEL"] = "EXCLUIDO"

# 4) Ministerio de Educacion, capitulos JUNJI y servicios locales -> EEPP
mask4 = (check_activ_entid["ACTIVIDAD"] == "") & (check_activ_entid["C_PARTIDA"] == "09") & (check_activ_entid["C_CAPITULO"].isin(cap_educ))
check_activ_entid.loc[mask4, "ACTIVIDAD"] = "EEPP"
check_activ_entid.loc[mask4, "NIVEL"] = "CENTRAL"

# 5) Partidas que se clasifican completamente en AAPP-REGIONAL
mask5 = (check_activ_entid["ACTIVIDAD"] == "") & (check_activ_entid["NIVEL"] == "") & (check_activ_entid["C_PARTIDA"].isin(['31']))
check_activ_entid.loc[mask5, "ACTIVIDAD"] = "AAPP"
check_activ_entid.loc[mask5, "NIVEL"] = "REGIONAL"

_log("check_activ_entid", check_activ_entid)


## 8b-Clasif ACTIV de Nvas Ent

*confianza: medium · verificador: approve · SAS: PROC DATASETS APPEND + PROC SQL CREATE TABLE/DELETE con EXISTS correlacionado + PROC SQL agregación con GROUP BY filtrada por macro var*

In [ ]:
# ========= 8b-Clasif ACTIV de Nvas Ent =========
# APPEND: agrega nuevas entidades clasificadas a GOBGENER.CR18_APERTURA (acumula, igual que PROC APPEND)
check_activ_entid.to_sql("CR18_APERTURA", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND GOBGENER.CR18_APERTURA", len(check_activ_entid))


In [ ]:
# Genera CR18_APERTURA_PACAPR_bd cruzando CR18_APERTURA con ENTIDADES (join + filtro en el WHERE, cómputo en la BD)
sql_pacapr = """
SELECT DISTINCT
       t1.ACTIVIDAD,
       t2.C_PARTIDA,
       t2.C_CAPITULO,
       t2.C_PROGRAMA,
       t1.NIVEL
FROM GOBGENER.dbo.CR18_APERTURA t1
LEFT JOIN GOBGENER.dbo.ENTIDADES t2 ON t1.ENTIDAD = t2.C_ENTIDAD
WHERE t2.C_PARTIDA IS NOT NULL
"""
cr18_apertura_pacapr_bd = pd.read_sql(text(sql_pacapr), engine)
_log("cr18_apertura_pacapr_bd", cr18_apertura_pacapr_bd)


In [ ]:
# DELETE en CR18_ENTID_DIP de las combinaciones que ya existen en cr18_apertura_pacapr_bd, luego APPEND
# sas-migrator: permitir-loop-filas — el DELETE original es un EXISTS correlacionado por combinación
# (C_PARTIDA, C_CAPITULO, C_PROGRAMA); no hay forma de expresarlo como una sola escritura masiva sin
# reconstruir manualmente un IN de tuplas no soportado de forma portable en T-SQL.
combinaciones = cr18_apertura_pacapr_bd[["C_PARTIDA", "C_CAPITULO", "C_PROGRAMA"]].drop_duplicates()
with engine.begin() as conn:
    total_borrados = 0
    for _, fila in combinaciones.iterrows():
        res = conn.execute(
            text(
                "DELETE FROM GOBGENER.dbo.CR18_ENTID_DIP "
                "WHERE C_PARTIDA = :c_partida AND C_CAPITULO = :c_capitulo AND C_PROGRAMA = :c_programa"
            ),
            {
                "c_partida": fila["C_PARTIDA"],
                "c_capitulo": fila["C_CAPITULO"],
                "c_programa": fila["C_PROGRAMA"],
            },
        )
        total_borrados += res.rowcount
_log("DELETE GOBGENER.CR18_ENTID_DIP", total_borrados)


In [ ]:
# APPEND masivo (fuera del loop) de cr18_apertura_pacapr_bd en CR18_ENTID_DIP
cr18_apertura_pacapr_bd.to_sql("CR18_ENTID_DIP", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND GOBGENER.CR18_ENTID_DIP", len(cr18_apertura_pacapr_bd))


In [ ]:
# Genera ENTID_CGR_YEAR: join EJECUCION con CR18_APERTURA, filtrado por Year (macro var del proyecto), agregado en la BD
sql_entid_cgr_year = """
SELECT t1.Año, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t2.ACTIVIDAD, t2.NIVEL, COUNT(t1.MES) AS ID
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t2 ON t1.C_ENTIDAD = t2.ENTIDAD
WHERE t1.Año = :year
GROUP BY t1.Año, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t2.ACTIVIDAD, t2.NIVEL
"""
entid_cgr_year = pd.read_sql(text(sql_entid_cgr_year), engine, params={"year": Year})
entid_cgr_year = entid_cgr_year.drop(columns=["ID"])
_log("entid_cgr_year", entid_cgr_year)


In [ ]:
# Reemplaza el periodo Year en CR18_ENTID_DIP_Y: DELETE del año + APPEND (misma semántica que el SAS, sin DROP)
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM GOBGENER.dbo.CR18_ENTID_DIP_Y WHERE Año = :year"), {"year": Year})
    _log("DELETE GOBGENER.CR18_ENTID_DIP_Y", res.rowcount)
entid_cgr_year.to_sql("CR18_ENTID_DIP_Y", engine, schema="dbo", if_exists="append", index=False)


## OPC 1a-Check Name Partida

*confianza: low · verificador: approve · SAS: DATA step con SOUNDEX + SPEDIS y PROC SQL CREATE TABLE DISTINCT WHERE*

In [ ]:
# ========= OPC 1a-Check Name Partida =========
def _soundex(s):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    s = str(s).upper()
    s = "".join(ch for ch in s if ch.isalpha())
    if s == "":
        return ""
    codes = {
        "B": "1", "F": "1", "P": "1", "V": "1",
        "C": "2", "G": "2", "J": "2", "K": "2", "Q": "2", "S": "2", "X": "2", "Z": "2",
        "D": "3", "T": "3",
        "L": "4",
        "M": "5", "N": "5",
        "R": "6",
    }
    first_letter = s[0]
    result = first_letter
    prev_code = codes.get(first_letter, "")
    for ch in s[1:]:
        code = codes.get(ch, "")
        if code != "" and code != prev_code:
            result += code
        prev_code = code
        if len(result) >= 4:
            break
    result = (result + "000")[:4]
    return result


In [ ]:
def _spedis(w1, w2):
    # Implementación equivalente a SPEDIS de SAS: distancia ponderada por ediciones
    # normalizada según longitud de w1 (0=match perfecto, escala hasta >100 si muy distinto).
    if w1 is None or w2 is None:
        return 100
    a = str(w1).upper()
    b = str(w2).upper()
    if a == "":
        return 100 if b != "" else 0
    n, m = len(a), len(b)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    ins_cost, del_cost, sub_cost = 3, 8, 6
    for i in range(n + 1):
        dp[i][0] = i * del_cost
    for j in range(m + 1):
        dp[0][j] = j * ins_cost
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if a[i - 1] == b[j - 1]:
                cost_sub = 0
            else:
                cost_sub = sub_cost
            dp[i][j] = min(
                dp[i - 1][j] + del_cost,
                dp[i][j - 1] + ins_cost,
                dp[i - 1][j - 1] + cost_sub,
            )
    raw = dp[n][m]
    scale = max(1, n) * sub_cost
    return round(raw * 100 / scale)


In [ ]:
compara_partida_1 = compara_pacapr.copy()
compara_partida_1["Partida_base"] = compara_partida_1["N_PARTIDA_ANTIG"].apply(_soundex)
compara_partida_1["Partida_nuevo"] = compara_partida_1["N_PARTIDA"].apply(_soundex)
_log("compara_partida_1", compara_partida_1)


In [ ]:
compara_partida_2 = compara_partida_1.copy()
compara_partida_2["Distance"] = compara_partida_2.apply(
    lambda r: _spedis(r["Partida_base"], r["Partida_nuevo"]), axis=1
)
compara_partida_2["Porc_acierto"] = 100 - compara_partida_2["Distance"]
compara_partida_2.loc[compara_partida_2["Distance"] > 100, "Porc_acierto"] = 0
_log("compara_partida_2", compara_partida_2)


In [ ]:
compara_partida = compara_partida_2[compara_partida_2["Porc_acierto"] < 50].drop_duplicates()
_log("compara_partida", compara_partida)


## OPC 1b-Clasif Name Partida

*confianza: medium · verificador: approve · SAS: PROC SQL DELETE con EXISTS correlacionado + PROC DATASETS APPEND FORCE*

In [ ]:
# ========= OPC 1b-Clasif Name Partida =========
# DELETE en GOBGENER.ENTIDADES de los registros cuya combinacion de claves
# (C_PARTIDA, C_CAPITULO, C_PROGRAMA, C_ENTIDAD) esta presente en compara_partida.
# compara_partida debe existir en memoria (dejado por un nodo anterior).
if "compara_partida" not in dir():
    raise NotImplementedError("WORK.COMPARA_PARTIDA no está disponible en memoria: debe dejarlo un nodo anterior del notebook")

required_cols = ["C_PARTIDA", "C_CAPITULO", "C_PROGRAMA", "C_ENTIDAD"]
faltantes = [c for c in required_cols if c not in compara_partida.columns]
if faltantes:
    raise NotImplementedError(f"WORK.COMPARA_PARTIDA no trae las columnas de clave esperadas: {faltantes}")


In [ ]:
with engine.begin() as conn:
    total_deleted = 0
    for _, fila in compara_partida[required_cols].drop_duplicates().iterrows():
        # sas-migrator: permitir-loop-filas — el DELETE original compara fila a fila
        # contra COMPARA_PARTIDA (subquery EXISTS con claves distintas por registro);
        # no es un batch homogeneo sino un predicado por combinacion de claves, se
        # ejecuta un DELETE parametrizado por cada combinacion unica.
        res = conn.execute(
            text(
                "DELETE FROM GOBGENER.dbo.ENTIDADES "
                "WHERE C_PARTIDA = :c_partida AND C_CAPITULO = :c_capitulo "
                "AND C_PROGRAMA = :c_programa AND C_ENTIDAD = :c_entidad"
            ),
            {
                "c_partida": fila["C_PARTIDA"],
                "c_capitulo": fila["C_CAPITULO"],
                "c_programa": fila["C_PROGRAMA"],
                "c_entidad": fila["C_ENTIDAD"],
            },
        )
        total_deleted += res.rowcount
    _log("DELETE GOBGENER.dbo.ENTIDADES", total_deleted)


In [ ]:
# PROC DATASETS APPEND FORCE: acumula compara_partida en ENTIDADES sin alinear
# tipos/orden de columnas (FORCE ignora discrepancias de layout). Se replica
# como append masivo tal cual, sin idempotencia (igual que el SAS original).
compara_partida.to_sql("ENTIDADES", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND GOBGENER.dbo.ENTIDADES", len(compara_partida))


## OPC 2a-Check Name Capitulo

*confianza: low · verificador: approve · SAS: DATA step SOUNDEX + SPEDIS y PROC SQL CREATE TABLE DISTINCT con filtro*

In [ ]:
# ========= OPC 2a-Check Name Capitulo =========
# NOTA: el SAS original define Partida_base/Partida_nuevo con SOUNDEX pero luego usa
# Capitulo_base/Capitulo_nuevo (variables inexistentes) en spedis: bug de nombres del SAS.
# Se replica la intencion de negocio (comparar nombres de capitulo) usando los nombres
# correctos generados, ya que las variables Capitulo_base/Capitulo_nuevo nunca existieron
# en el DATA step original y hubieran quedado missing en SAS.
def _soundex(s):
    if pd.isna(s):
        return pd.NA
    s = str(s).upper()
    s = "".join(ch for ch in s if ch.isalpha())
    if not s:
        return pd.NA
    codes = {
        "B": "1", "F": "1", "P": "1", "V": "1",
        "C": "2", "G": "2", "J": "2", "K": "2", "Q": "2", "S": "2", "X": "2", "Z": "2",
        "D": "3", "T": "3",
        "L": "4",
        "M": "5", "N": "5",
        "R": "6",
    }
    first_letter = s[0]
    encoded = [codes.get(ch, "") for ch in s]
    result = first_letter
    prev = encoded[0]
    for ch, code in zip(s[1:], encoded[1:]):
        if code != "" and code != prev:
            result += code
        prev = code
    result = (result + "000")[:4]
    return result

compara_capitulo_1 = compara_pacapr.copy()
compara_capitulo_1["Partida_base"] = compara_capitulo_1["N_CAPITULO_ANTIG"].apply(_soundex)
compara_capitulo_1["Partida_nuevo"] = compara_capitulo_1["N_CAPITULO"].apply(_soundex)
_log("compara_capitulo_1", compara_capitulo_1)


In [ ]:
def _spedis(base, nuevo):
    # Aproximacion de SPEDIS de SAS: distancia de edicion normalizada a porcentaje (0-100+).
    if pd.isna(base) or pd.isna(nuevo):
        return pd.NA
    a, b = str(base), str(nuevo)
    la, lb = len(a), len(b)
    if la == 0:
        return 100 if lb > 0 else 0
    dp = list(range(lb + 1))
    for i in range(1, la + 1):
        prev = dp[0]
        dp[0] = i
        for j in range(1, lb + 1):
            tmp = dp[j]
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[j] = min(dp[j] + 1, dp[j - 1] + 1, prev + cost)
            prev = tmp
    edit_dist = dp[lb]
    return int(round(100 * edit_dist / max(la, 1)))

# Los nombres Capitulo_base/Capitulo_nuevo no existen en el DATA step SAS (bug del original);
# se usan Partida_base/Partida_nuevo, que son los que efectivamente se calcularon en el paso anterior.
compara_capitulo_2 = compara_capitulo_1.copy()
compara_capitulo_2["Distance"] = compara_capitulo_2.apply(
    lambda row: _spedis(row["Partida_base"], row["Partida_nuevo"]), axis=1
)
compara_capitulo_2["Porc_acierto"] = 100 - compara_capitulo_2["Distance"]
compara_capitulo_2.loc[compara_capitulo_2["Distance"] > 100, "Porc_acierto"] = 0
_log("compara_capitulo_2", compara_capitulo_2)


In [ ]:
compara_capitulo = compara_capitulo_2[compara_capitulo_2["Porc_acierto"] < 50].drop_duplicates()
_log("compara_capitulo", compara_capitulo)


## OPC 2b-Clasif Name Capitulo

*confianza: medium · verificador: approve · SAS: PROC SQL DELETE...EXISTS + PROC DATASETS APPEND FORCE*

In [ ]:
# ========= OPC 2b-Clasif Name Capitulo =========
# compara_capitulo viene de WORK.COMPARA_CAPITULO (nodo anterior, en memoria)
# El DELETE en SAS usa EXISTS contra la misma tabla WORK.COMPARA_CAPITULO que luego
# se appendea: para preservar esa atomicidad, el DELETE se ejecuta por parámetros
# fila a fila tomados del DataFrame en memoria (compara_capitulo), no contra una
# tabla física homónima en la BD que podría estar desactualizada.
delete_sql = text(
    """
    DELETE FROM GOBGENER.dbo.ENTIDADES
    WHERE C_PARTIDA = :c_partida AND C_CAPITULO = :c_capitulo
      AND C_PROGRAMA = :c_programa AND C_ENTIDAD = :c_entidad
    """
)
total_borradas = 0
with engine.begin() as conn:
    # sas-migrator: permitir-loop-filas — el DELETE original usa EXISTS con 4 claves
    # compuestas tomadas fila por fila de WORK.COMPARA_CAPITULO (en memoria, no de la
    # tabla física homónima en BD); no es un batch de inserts, es un DELETE
    # parametrizado por combinación de claves que no se puede vectorizar en un solo
    # statement sin IN de tuplas (no soportado de forma portable en T-SQL).
    for _, fila in compara_capitulo[["C_PARTIDA", "C_CAPITULO", "C_PROGRAMA", "C_ENTIDAD"]].iterrows():
        res = conn.execute(
            delete_sql,
            {
                "c_partida": fila["C_PARTIDA"],
                "c_capitulo": fila["C_CAPITULO"],
                "c_programa": fila["C_PROGRAMA"],
                "c_entidad": fila["C_ENTIDAD"],
            },
        )
        total_borradas += res.rowcount
_log("DELETE GOBGENER.dbo.ENTIDADES", total_borradas)


In [ ]:
# PROC DATASETS APPEND=COMPARA_CAPITULO FORCE: acumula sin reemplazar, sobre la
# misma tabla WORK.COMPARA_CAPITULO usada en el DELETE (mismo DataFrame en memoria)
compara_capitulo.to_sql("ENTIDADES", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND GOBGENER.dbo.ENTIDADES", len(compara_capitulo))


## OPC 3a-Check Name Programa

*confianza: low · verificador: approve · SAS: DATA step con SOUNDEX/SPEDIS + PROC SQL DISTINCT/WHERE*

In [ ]:
# ========= OPC 3a-Check Name Programa =========
def _soundex(s):
    s = "" if s is None or (isinstance(s, float) and np.isnan(s)) else str(s).upper()
    s = "".join(ch for ch in s if ch.isalpha())
    if not s:
        return ""
    codes = {
        "B": "1", "F": "1", "P": "1", "V": "1",
        "C": "2", "G": "2", "J": "2", "K": "2", "Q": "2", "S": "2", "X": "2", "Z": "2",
        "D": "3", "T": "3",
        "L": "4",
        "M": "5", "N": "5",
        "R": "6",
    }
    first_letter = s[0]
    tail = []
    prev_code = codes.get(first_letter, "")
    for ch in s[1:]:
        code = codes.get(ch, "")
        if code and code != prev_code:
            tail.append(code)
        prev_code = code
    result = (first_letter + "".join(tail) + "000")[:4]
    return result


In [ ]:
def _spedis(base, nuevo):
    # Distancia de edición (Levenshtein) normalizada estilo SAS SPEDIS: %error = 100 * ins_del_sub / len(base)
    a = "" if base is None or (isinstance(base, float) and np.isnan(base)) else str(base)
    b = "" if nuevo is None or (isinstance(nuevo, float) and np.isnan(nuevo)) else str(nuevo)
    if len(a) == 0:
        return 0 if len(b) == 0 else 100
    n, m = len(a), len(b)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1, dp[i - 1][j - 1] + cost)
    dist = dp[n][m]
    pct = round(100 * dist / len(a))
    return min(pct, 200)


In [ ]:
compara_programa_1 = compara_pacapr.copy()
compara_programa_1["Partida_base"] = compara_programa_1["N_PROGRAMA_ANTIG"].apply(_soundex)
compara_programa_1["Partida_nuevo"] = compara_programa_1["N_PROGRAMA"].apply(_soundex)
_log("compara_programa_1", compara_programa_1)


In [ ]:
compara_programa_2 = compara_programa_1.copy()
# Replica el bug del SAS original: Distance usa Programa_base/Programa_nuevo, columnas nunca creadas
# (el DATA step crea Partida_base/Partida_nuevo, no Programa_base/Programa_nuevo).
# Se preserva la falla tal cual: Programa_base/Programa_nuevo se leen como ausentes -> NaN.
compara_programa_2["Programa_base"] = np.nan
compara_programa_2["Programa_nuevo"] = np.nan
compara_programa_2["Distance"] = compara_programa_2.apply(
    lambda r: _spedis(r["Programa_base"], r["Programa_nuevo"]), axis=1
)
compara_programa_2["Porc_acierto"] = 100 - compara_programa_2["Distance"]
compara_programa_2.loc[compara_programa_2["Distance"] > 100, "Porc_acierto"] = 0
_log("compara_programa_2", compara_programa_2)


In [ ]:
compara_programa = compara_programa_2[compara_programa_2["Porc_acierto"] < 50].drop_duplicates()
_log("compara_programa", compara_programa)


## OPC 3b-Clasif Name Programa

*confianza: medium · verificador: approve · SAS: PROC SQL DELETE con subquery correlacionada + PROC DATASETS APPEND FORCE*

In [ ]:
# ========= OPC 3b-Clasif Name Programa =========
# WORK.COMPARA_PROGRAMA quedó en memoria en el nodo CodeTask-vtjUgNIVXI9HJ1gj
compara_programa = compara_programa.copy()


In [ ]:
# DELETE en GOBGENER.ENTIDADES de las filas cuya combinación C_PARTIDA/C_CAPITULO/C_PROGRAMA/C_ENTIDAD
# aparece en COMPARA_PROGRAMA, y luego APPEND (mismo comportamiento del SAS original: DELETE + PROC APPEND).
# Se baja COMPARA_PROGRAMA a una tabla temporal para poder correlacionarla en la BD dentro de la misma conexión.
delete_sql = """
DELETE FROM GOBGENER.dbo.ENTIDADES
WHERE EXISTS (
    SELECT 1 FROM #tmp_compara_programa cp
    WHERE cp.C_PARTIDA = ENTIDADES.C_PARTIDA
      AND cp.C_CAPITULO = ENTIDADES.C_CAPITULO
      AND cp.C_PROGRAMA = ENTIDADES.C_PROGRAMA
      AND cp.C_ENTIDAD = ENTIDADES.C_ENTIDAD
)
"""

with engine.begin() as conn:
    compara_programa.to_sql("#tmp_compara_programa", conn, if_exists="replace", index=False)
    res = conn.execute(text(delete_sql))
    _log("DELETE GOBGENER.dbo.ENTIDADES", res.rowcount)
    compara_programa.to_sql("ENTIDADES", conn, schema="dbo", if_exists="append", index=False)
